# Experiment 04 — score matrices with v6-compatible options

Based on [01-initial.ipynb](01-initial.ipynb), [03-backtest-alternative-weighting.ipynb](03-backtest-alternative-weighting.ipynb) and the first experiment 04. This revision updates experiment 04 in place as requested. The imported `hyper-ai-v11.py` supplies the unchanged universe, selection machinery, capacity redistribution, guards, fees and trade execution.

## User goal

Calculate per-vault Sharpe, Sortino, max drawdown, CAGR and Calmar across multiple windows. Rank by a period × metric matrix, then compare equal and score-proportional position weights. Optimise portfolio CAGR and Sharpe separately, blend their scores, and investigate fewer than five active weights. Original requested clips are CAGR 50%, Sharpe 3, Sortino 7 and drawdown −70% with inverse scoring; Calmar starts at 5 with neighbour diagnostics.

**Latest requested changes:** include a **46-day cell**, replace the **365-day feature window with 360 days**, include a **60/40 blend**, add **90-day inverse-variance sizing**, and test a **`missing` boolean flag** reproducing v6's unscored behaviour. Keep selection machinery unchanged. **Fit and retain the long-period search**, and prove before searching that the variant machinery can express v6 scoring/sizing or very close rules.

## Report period policy

Every displayed chart and performance table, including the Markdown summary and attribution, uses **2026-01-01 → 2026-10-05 (end exclusive, UTC)** by default. These are fresh replays with \$150,000 and no inherited positions. Data before January remains available for rolling indicators. The sparse search is repeated on the repaired inputs and its coefficients are then frozen from the full-history fit; that fitting period overlaps 2026, so the report is not independent validation. Parameter tables describe the frozen inputs used in these replays; they are not performance results. Full-history evidence is retained in generated artefacts rather than displayed as the default report. This authorised refresh pins the repaired 7 October inputs and uses a new indicator and result generation. Default execution reruns against that pinned snapshot; saved-report mode can be enabled for presentation-only rebuilds.

## Method and exact implementation

### Compatibility must pass before optimisation

The source v6 formula is 60% clipped 360-day CAGR plus 40% clipped 45-day Sortino, with clips **100% / 3**, 365-day annualisation, and position requests proportional to **1 / max(90-day daily standard deviation, 0.0001)^2**. The notebook retains a 45-day compatibility cell solely to verify this formula; the searchable grid uses the requested 46-day cell. All vault/date scores, missing-value patterns and sizing requests are compared directly with v6 functions. The exact-formula matrix then has to reproduce the original v11 control's complete trade ledger and equity curve. A failed check stops the notebook before the search.

These checks establish **v6 scoring/sizing under v11 selection**, not an exact native-v6 strategy replay. V11 hysteresis (entry N, exit N+2), deterministic address ties, deposit/guard checks and core/capacity rules remain as in the earlier experiment. No source strategy or framework files are edited.

### Periods, clipping and missingness

Searchable periods are **30, 46, 60, 90, 120, 180, 270, 360 and 730 days**: 45 cells. Five extra 45-day cells support compatibility only and are excluded from sparse feature proposals. A D-day metric needs D returns and D+1 positive observations. Feature T ends on **T−1**. Annualisation stays **365**; replacing the one-year lookback with 360 does not change annualisation. Existing carried NAV remains carried NAV.

Matrix weights are non-negative and sum to one. The **requested** profile uses clips 50% / 3 / 7 / 70% / 5 for CAGR / Sharpe / Sortino / drawdown magnitude / Calmar. The explicit **legacy** profile changes only CAGR/Sortino clips to 100% / 3. Calmar uses a 1% drawdown denominator floor. Drawdown quality is `1 − min(abs(DD), 0.70)/0.70`.

- **`missing=False`:** unavailable cells contribute zero; Sharpe/Sortino use the previous 0.0001 daily denominator floor. No available-period renormalisation. Zero total score gets zero requested allocation, as in the first experiment 04.
- **`missing=True`:** missing history or an undefined active metric makes the whole matrix score NaN. Sharpe/Sortino use actual risk, and zero denominators yield NaN as in v6. Inactive cells cannot poison the score. V11 converts NaN to rank score zero. An unscored vault remains a lowest-priority candidate: equal and inverse-variance sizing can still fund it if selected, and composite sizing retains the source's all-zero fallback. This reproduces v6 semantics without adding a selection veto.

Sizing methods are **equal**, **score** (the source composite method), and **inverse_variance**, using v11's existing 90-day inverse-volatility indicator squared. All retain `$150k` starting equity, 98% deployment budget, 33% concentration/TVL caps, the 14-day −16% return gate, two-day decisions, the source modelled 10-basis-point redemption charge, trade bands, `$5` deposit minimum and 0.5% basket cleanup. For N=4/6/10, retain the existing entry N/exit N+2 settings and core=capacity=N.

### Search and comparisons

1. Pass the compatibility checks and original v11 long replay replication before searching.
2. Screen the 45 searchable singletons with both missing flags, at six equal-weight positions. Also seed explicit **60% 360-day CAGR / 40% 46-day Sortino** matrices under both clipping profiles, all three sizing methods and both flags. These anchors are candidates; the optimiser cannot omit them by chance.
3. Run **18 seeded sparse proposals per objective** and fixed-support ±0.10 refinements. Supports have two through four cells; additions come from the best twelve singletons, with a full-singleton fallback. Proposal sizing cycles through equal, score and inverse variance; the missing flag alternates. New-feature masses include 40% and 60%. This is a bounded initial search, not exhaustive optimisation.
4. Freeze CAGR and Sharpe winners, blend their **actual vault scores** 50/50 and **60/40 (CAGR/Sharpe)**, and test top-weight compression of the 50/50 blend to one through four cells. Compression passes within 1 percentage point CAGR and 0.10 Sharpe on the reference full-period run. Both blends preserve each winner’s own clip profile. Their scores, rather than just their coefficients, are averaged. Missingness and position sizing are then compared as shared settings. Compression discards selected cells across both component profiles and renormalises their retained coefficient mass.
5. Compare seven fixed families (CAGR winner, Sharpe winner, 50/50, 60/40, compressed, dense, explicit 360/46 anchor) at **4/6/10 positions × three sizing methods × both missing flags**. Include the legacy-clipped 360/46 anchor's six-position diagnostics and the exact 360/45 compatibility control. Keep selected family results, anchors, original control and three accepted-dollar-cap diagnostics for charts and position attribution.
6. Calmar caps 3/5/10 are frozen six-position sensitivities, not a new weight search. Report key metrics side by side, equity, quarterly compounded profit heatmaps and absolute USD allocation.

### Interpretation and reproducibility

The fitting and full-history comparison period is **all available vault history, 1 March 2023–5 October 2026, end exclusive**, plus the engine's initial 28 February cash snapshot. Coefficients and detailed settings were selected on that long window: **there is no independent validation or holdout**, and rankings are exploratory. The previous revision's short/validation findings are superseded, not carried forward as new findings. Current-universe and guard snapshots remain historical assumptions; early deposit-state history is unavailable. Two-year coverage and flat marks need explicit audits.

Matching caches are keyed by metric panel, source/guard/framework hashes and configuration, including missing flag and clip profile. Generated artefacts stay ignored. Position/vault PnL must reconcile to equity. Largest days, kurtosis, curator flags and Binance BTC/ETH over matching snapshot intervals are inspected. Best-day-zero and leading-vault removal retain the original trading path and are sensitivities, not exclusion backtests.

## Reading the visual report

Start with the coloured key metrics, then the singleton and full-grid heatmaps. CAGR and Sharpe feature-correlation heatmaps show redundancy across lookbacks and the effect of clipping. The paired missing-flag chart isolates that flag; coefficient fingerprints explain matrix structure. The CAGR/Sharpe frontier and interactive parallel-coordinate explorer show trade-offs and let you inspect subsets. Equity, quarterly returns and absolute allocations explain the surviving paths. Every chart and table has a subheading, a short description and then the element. Titles and captions state the reporting period.

Repeated paths are shown once and supporting logs/tables are compact or collapsed. Full generated evidence is retained. Execution progress uses one updating status line per stage; warnings and failures remain visible. The visual changes do not alter any scoring, search, selection or execution rules.

## Key new insights

- **Default period:** every displayed performance result is a fresh 2026-01-01 → 2026-10-05 (end exclusive, UTC) replay with \$150,000 and no inherited positions.
- **Original v11:** 41.20% CAGR / 1.96 Sharpe / -6.40% max drawdown. Exact 360/45 compatibility reproduces its full equity path and all 581 trades.
- **Near-v6 360/46:** 40.58% CAGR / 1.94 Sharpe / -6.70% max drawdown. Its score window differs from the exact control by one day.
- **Highest observed grid CAGR:** 41.38% for 50/50 blend, 6 positions, inverse_variance, missing=True; Sharpe 2.12, drawdown -4.85%. This grid maximum does not change the frozen strategy settings.
- **Joint improvements:** 1 distinct qualifying configuration with higher CAGR, higher Sharpe and no worse drawdown than original v11 in 2026. The fitting period overlaps this replay, so these are in-sample observations.
- **Repaired-input benchmark:** Archived leader-share inputs and policy caps are restored. Source v11 reaches 41.20% CAGR versus the older 41.59%; its first 558 chronological trades match. The final equity gap is \$407.83 and is reconciled in restored-data-rerun-2026-10-07.md.
- **Feature redundancy:** same-lookback CAGR and Sharpe scores have Spearman correlations of 0.89–0.92 under requested clipping in 2026. The heatmaps compare raw metrics, clipping and lookbacks with pair counts; these are pooled feature correlations, not evidence of predictive performance.

## Summary of results

### Frozen matrices used in the replay — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Input coefficients were selected on full history and are held fixed for 2026. Identical coefficients with different clipping recipes can produce different scores; these rows are parameters, not performance.

| Family | Clips | Active cells | Period / metric weights |
| --- | --- | --- | --- |
| CAGR matrix | requested | 2 | 1y CAGR: 60%; 46d Sortino: 40% |
| Sharpe matrix | legacy | 2 | 1y CAGR: 60%; 46d Sortino: 40% |
| 50/50 blend | 50% requested + 50% legacy | 2 | 1y CAGR: 60%; 46d Sortino: 40% |
| 60/40 blend | 60% requested + 40% legacy | 2 | 1y CAGR: 60%; 46d Sortino: 40% |
| Compressed blend | 50% requested + 50% legacy | 2 | 1y CAGR: 60%; 46d Sortino: 40% |
| Dense equal matrix | requested | 45 | All 45 searchable cells equally weighted |
| 360/46 anchor | requested | 2 | 1y CAGR: 60%; 46d Sortino: 40% |

### Strategy performance side by side — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare distinct fresh 2026 equity/trade paths using the long-fitted strategy settings. Equivalent named aliases appear once. Drawdown is signed: values nearer zero are better.

| Family | Positions | Sizing | Missing | Clips | Allocator | Final equity (USD) | CAGR | Sharpe | Sortino | Max DD | Mean cash weight | Trades |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Original v11 | 6 | inverse_variance | True | legacy | parent | \$194,521 | 41.20% | 1.96 | 3.43 | -6.40% | 8.83% | 581 |
| Near-v6 360/46 legacy | 6 | inverse_variance | True | legacy | parent | \$193,876 | 40.58% | 1.94 | 3.39 | -6.70% | 8.99% | 572 |
| CAGR matrix | 6 | equal | False | requested | parent | \$154,762 | 4.35% | 0.32 | 0.45 | -12.38% | 10.83% | 597 |
| 50/50 blend | 6 | equal | False | 50% requested + 50% legacy | parent | \$183,564 | 30.75% | 1.50 | 2.28 | -9.47% | 11.80% | 554 |
| 60/40 blend | 6 | equal | False | 60% requested + 40% legacy | parent | \$176,236 | 23.89% | 1.23 | 1.84 | -9.31% | 11.56% | 564 |
| Dense equal matrix | 4 | score | True | requested | parent | \$152,155 | 2.00% | 0.27 | 0.35 | -8.06% | 8.89% | 308 |
| CAGR matrix | 6 | equal | False | requested | accepted_dollar_cap | \$149,550 | -0.29% | 0.07 | 0.09 | -11.76% | 19.22% | 496 |
| 60/40 blend | 6 | equal | False | 60% requested + 40% legacy | accepted_dollar_cap | \$170,188 | 18.30% | 1.07 | 1.57 | -8.28% | 21.27% | 472 |
| Sharpe matrix | 6 | inverse_variance | True | legacy | accepted_dollar_cap | \$159,117 | 8.20% | 1.23 | 1.67 | -3.79% | 52.16% | 462 |

### Explicit 60/40 anchor options — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Hold six positions and ranking at 60% 360-day CAGR plus 40% 46-day Sortino. Compare only clipping profiles, sizing and missing flags in the short replay.

| Clips | Sizing | Missing | CAGR | Sharpe | Max DD | Mean cash weight |
| --- | --- | --- | --- | --- | --- | --- |
| requested | equal | False | 4.35% | 0.32 | -12.38% | 10.83% |
| requested | equal | True | 8.33% | 0.52 | -12.56% | 11.86% |
| requested | score | False | 2.23% | 0.21 | -12.68% | 9.89% |
| requested | score | True | 6.61% | 0.43 | -12.66% | 10.77% |
| requested | inverse_variance | False | 23.05% | 1.53 | -4.23% | 5.71% |
| requested | inverse_variance | True | 28.09% | 1.77 | -4.07% | 5.88% |
| legacy | equal | False | 23.71% | 1.14 | -9.95% | 11.89% |
| legacy | equal | True | 27.99% | 1.31 | -10.03% | 12.62% |
| legacy | score | False | 20.79% | 0.97 | -10.88% | 10.66% |
| legacy | score | True | 25.31% | 1.15 | -10.77% | 11.22% |
| legacy | inverse_variance | False | 35.81% | 1.77 | -6.79% | 8.68% |
| legacy | inverse_variance | True | 40.58% | 1.94 | -6.70% | 8.99% |

### Rolling metric availability — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Coverage counts complete positive-price windows across all vaults × replay dates, including dates before a vault has enough history. Undefined ratios use only complete windows as their denominator; flat windows are vault-day counts. Lookback labels are indicator windows, not reporting periods.

| Period | Coverage | Undefined ratio within complete windows | Flat complete vault-days |
| --- | --- | --- | --- |
| 1m | 81.94% | 27.76% | 30729 |
| 46d | 79.08% | 24.30% | 26320 |
| 2m | 76.46% | 21.95% | 23006 |
| 3m | 70.48% | 17.54% | 16873 |
| 4m | 64.05% | 13.51% | 11683 |
| 6m | 50.31% | 6.38% | 4007 |
| 9m | 34.04% | 2.75% | 996 |
| 1y | 22.78% | 2.33% | 646 |
| 2y | 3.73% | 0.98% | 0 |

### Profit concentration and return sensitivities — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Identify the leading vault and strongest booked day in each 2026 path. Removing their PnL retains the original trades and is a sensitivity, not a new executable backtest.

| Run | Leading vault | Leading vault profit (USD) | Leading vault share of positive vault profits | Best day | Best day return | Sharpe with best day set to zero | Sharpe without leader PnL (fixed ledger) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 50/50 blend / parent | Realist Capital | \$13,980 | 26.49% | 2026-06-26 | 4.37% | 1.24 | 0.98 |
| 60/40 blend / parent | Realist Capital | \$12,424 | 27.43% | 2026-06-26 | 4.58% | 0.94 | 0.74 |
| CAGR matrix / parent | Realist Capital | \$8,669 | 29.75% | 2026-06-26 | 6.40% | -0.15 | -0.12 |
| Dense equal matrix / parent | Hyperliquidity Provider (HLP) | \$2,234 | 39.90% | 2026-01-07 | 3.06% | -0.21 | 0.05 |
| Original v11 / parent | Realist Capital | \$24,196 | 44.77% | 2026-06-26 | 6.89% | 1.62 | 1.10 |
| Near-v6 360/46 legacy / parent | Realist Capital | \$25,883 | 48.48% | 2026-06-26 | 6.87% | 1.60 | 0.99 |
| CAGR matrix / accepted_dollar_cap | Goon Edging | \$5,187 | 22.78% | 2026-06-26 | 4.16% | -0.28 | -0.25 |
| Sharpe matrix / accepted_dollar_cap | pmalt | \$3,824 | 28.42% | 2026-02-28 | 1.87% | 0.90 | 0.75 |
| 60/40 blend / accepted_dollar_cap | Realist Capital | \$8,789 | 23.10% | 2026-06-26 | 4.16% | 0.78 | 0.68 |

### Information available when funding positions — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fractions weight target dollars across 2026 decisions, not realised time-weighted exposure. Unscored funding can occur under missing=True. Original v11 is absent because it has no matrix allocation ledger.

| Run | Funded target rows | Unscored funding | Missing 360d history | Missing 2y history | Flat 1m marks |
| --- | --- | --- | --- | --- | --- |
| 50/50 blend / parent | 834 | 0.00% | 0.00% | 74.66% | 0.00% |
| 60/40 blend / parent | 834 | 0.00% | 0.00% | 74.33% | 0.00% |
| CAGR matrix / parent | 834 | 0.00% | 0.00% | 74.87% | 0.00% |
| Dense equal matrix / parent | 555 | 0.00% | 0.00% | 0.00% | 0.00% |
| Near-v6 360/46 legacy / parent | 834 | 0.00% | 0.00% | 73.79% | 0.00% |
| CAGR matrix / accepted_dollar_cap | 834 | 0.00% | 0.00% | 76.63% | 0.00% |
| Sharpe matrix / accepted_dollar_cap | 834 | 0.00% | 0.00% | 64.03% | 0.00% |
| 60/40 blend / accepted_dollar_cap | 834 | 0.00% | 0.00% | 76.71% | 0.00% |

### Compression candidates — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare completed short-period compression replays at six equal-weight positions and missing=False. The diagnostic tolerance is within one percentage point CAGR and 0.10 Sharpe of the full blend; the frozen compression choice is not reselected.

| Active cells | CAGR | Sharpe | Max DD | Within 2026 tolerance |
| --- | --- | --- | --- | --- |
| 1 | 24.60% | 1.04 | -13.58% | False |
| 2 | 30.75% | 1.50 | -9.47% | True |

### Calmar clipping sensitivity — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare caps 3/5/10 using six positions and the frozen sizing, missing flag and score recipe. These matrices have no active Calmar term, so changing the cap cannot alter their score or trade path.

| Family | Calmar cap | CAGR | Sharpe | Max DD |
| --- | --- | --- | --- | --- |
| CAGR matrix | 3.0 | 4.35% | 0.32 | -12.38% |
| CAGR matrix | 5.0 | 4.35% | 0.32 | -12.38% |
| CAGR matrix | 10.0 | 4.35% | 0.32 | -12.38% |
| Sharpe matrix | 3.0 | 40.58% | 1.94 | -6.70% |
| Sharpe matrix | 5.0 | 40.58% | 1.94 | -6.70% |
| Sharpe matrix | 10.0 | 40.58% | 1.94 | -6.70% |
| 50/50 blend | 3.0 | 30.75% | 1.50 | -9.47% |
| 50/50 blend | 5.0 | 30.75% | 1.50 | -9.47% |
| 50/50 blend | 10.0 | 30.75% | 1.50 | -9.47% |
| 60/40 blend | 3.0 | 23.89% | 1.23 | -9.31% |
| 60/40 blend | 5.0 | 23.89% | 1.23 | -9.31% |
| 60/40 blend | 10.0 | 23.89% | 1.23 | -9.31% |

## Robustness and interpretation

All 26 code cells completed. The short-period manifest records 194 configuration evaluations. Position PnL reconciles to equity minus starting capital, with a largest residual of 4.73e-11 USD. Source scoring, selection and execution rules are unchanged. Full-history fitting and reference artefacts remain available separately.

The long fitting window includes 2026: this is exploratory in-sample analysis, not independent validation. Frozen families and detailed settings are not reselected using the short-period rankings. Missing history, flat NAV marks, current-universe assumptions and incomplete historical deposit-state data remain limitations.


### Restored leader history rerun — 7 October 2026

This execution uses the freshly downloaded, pinned repaired dataset and the merged leader-share and recorded-cap readers. Strategy settings, searches and period boundaries are unchanged. Historical benchmark differences are displayed separately from mandatory source/control equality on the same inputs. Generated results are isolated under `_artifacts-restored-2026-10-07/04`.


In [1]:
from pathlib import Path
import sys
RESEARCH_DIR = Path("/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
sys.path.insert(0, str(RESEARCH_DIR))
from score_matrix_visualisation import (
    compact_output, chart_intro, present_table, show_static, with_backtest_period, collapsed_table, metric_table, search_table,
    profile_name, unique_factorial, unique_runs, run_title,
    singleton_heatmaps, factorial_heatmap, missing_effect_heatmaps,
    weight_fingerprints, pareto_chart, parallel_coordinates, FAMILY_COLOURS,
    quarterly_colour_limit, style_quarterly_returns, allocation_colours, style_allocation,
)
with compact_output('Notebook setup'):
    import datetime
    from copy import deepcopy
    import hashlib
    import importlib.util
    import logging
    import os
    import sys
    from pathlib import Path

    import pandas as pd
    import plotly.express as px
    from IPython.display import display
    from dotenv import load_dotenv

    from tradingstrategy.client import Client
    from tradeexecutor.backtest.backtest_runner import run_backtest_inline
    from tradeexecutor.strategy.execution_context import notebook_execution_context
    from tradeexecutor.strategy.parameters import StrategyParameters
    from tradeexecutor.strategy.reserve_currency import ReserveCurrency
    from tradeexecutor.statistics.key_metric import (
        calculate_cagr,
        calculate_max_drawdown,
        calculate_sharpe,
        calculate_sortino,
    )
    from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
    from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode

    load_dotenv(override=True)
    setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)

    STRATEGY_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v11.py")
    GUARD_PATH = Path("/Users/moo/code/strategies/deploy/hyper-ai-v10-new-guard-vault-info.json")
    os.environ["HYPER_AI_GUARD_CONFIG_FILE"] = str(GUARD_PATH)
    assert STRATEGY_PATH.is_file(), STRATEGY_PATH
    assert GUARD_PATH.is_file(), GUARD_PATH

    spec = importlib.util.spec_from_file_location("hyper_ai_v11", STRATEGY_PATH)
    strategy = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(strategy)
    source_sha256 = hashlib.sha256(STRATEGY_PATH.read_bytes()).hexdigest()
    parameters = StrategyParameters.from_class(strategy.Parameters)
    chart_start = datetime.datetime(2026, 1, 1)
    chart_end = parameters.backtest_end
    CHART_PERIOD = f"{chart_start:%Y-%m-%d} → {chart_end:%Y-%m-%d} (end exclusive, UTC)"
    client = Client.create_jupyter_client()

    present_table(pd.DataFrame([
        {"Input": "Strategy source", "Value": str(STRATEGY_PATH)},
        {"Input": "Strategy SHA-256", "Value": source_sha256},
        {"Input": "Guard record", "Value": str(GUARD_PATH)},
        {"Input": "Decision cycle", "Value": str(parameters.cycle_duration)},
        {"Input": "Initial equity (USD)", "Value": parameters.initial_cash},
    ]), "Strategy inputs", "Source module, guard record, decision cycle and starting capital used for the 2026 replays.", CHART_PERIOD, collapsed=True)

    RESEARCH_DIR = Path("/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
    sys.path.insert(0, str(RESEARCH_DIR))
    from accepted_dollar_cap import AuditedAlphaModel, make_decide_trades, calculate_position_profit_curves
    from tradeexecutor.analysis.weights import calculate_asset_weights
    from tqdm_loggable.auto import tqdm


    from score_matrix import MatrixConfig, ScorePanel, MatrixBacktester, FEATURES, SEARCH_FEATURES, PERIODS, weight_table, development_result, sparse_search
    import numpy as np
    import json
    from score_matrix_blends import register_score_blend, truncate_score_blend

    class SearchParameters:
        iterations = 18
        seed = 104
        position_counts = (4, 6, 10)
        sizing_methods = ("equal", "score", "inverse_variance")
        missing_values = (False, True)

    REUSE_SAVED_EXPERIMENT = False
    config = MatrixConfig()
    ARTIFACT_DIR = RESEARCH_DIR / "_artifacts-restored-2026-10-07/04"
    ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
    if REUSE_SAVED_EXPERIMENT:
        from score_matrix_report import SavedExperiment
        saved_experiment = SavedExperiment(ARTIFACT_DIR,parameters)
    def matrix(**cells):
        w = np.zeros(len(FEATURES))
        for key,value in cells.items():
            period,metric = key.split("|")
            w[FEATURES.index((period,metric))] = value
        assert abs(w.sum()-1)<1e-12
        return w
    legacy45 = matrix(**{"1y|CAGR":.6,"45d compatibility|Sortino":.4})
    anchor46 = matrix(**{"1y|CAGR":.6,"46d|Sortino":.4})
    dense = np.zeros(len(FEATURES))
    dense[SEARCH_FEATURES] = 1/len(SEARCH_FEATURES)
    present_table(pd.DataFrame([config.__dict__]), "Metric clipping and risk floors", "Fixed score caps and denominator floors applied to rolling vault metrics. Values are fractions or ratios, not portfolio returns.", CHART_PERIOD)
    present_table(pd.DataFrame([{"Iterations per fitting objective":18,"Search cells":len(SEARCH_FEATURES),"Positions":str(SearchParameters.position_counts),"Sizing":str(SearchParameters.sizing_methods),"Missing":str(SearchParameters.missing_values)}]), "Search design and replay grid", "The matrices were fitted on full history; the 2026 report compares their fixed coefficients across these position counts, sizing methods and missing flags.", CHART_PERIOD)


Notebook setup — complete

### Strategy inputs — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Source module, guard record, decision cycle and starting capital used for the 2026 replays.

Input,Value
Strategy source,/Users/moo/code/strategies/strategy/hyper-ai-v11.py
Strategy SHA-256,5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe
Guard record,/Users/moo/code/strategies/deploy/hyper-ai-v10-new-guard-vault-info.json
Decision cycle,CycleDuration.cycle_2d
Initial equity (USD),150000


### Metric clipping and risk floors — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fixed score caps and denominator floors applied to rolling vault metrics. Values are fractions or ratios, not portfolio returns.

,cagr_cap,sharpe_cap,sortino_cap,drawdown_cap,calmar_cap,calmar_drawdown_floor,daily_risk_floor
0,0.5,3.0,7.0,0.7,5.0,0.01,0.0001


### Search design and replay grid — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The matrices were fitted on full history; the 2026 report compares their fixed coefficients across these position counts, sizing methods and missing flags.

,Iterations per fitting objective,Search cells,Positions,Sizing,Missing
0,18,45,"(4, 6, 10)","('equal', 'score', 'inverse_variance')","(False, True)"


## Universe and operational indicators

Load the original v11 inputs and its existing 90-day inverse-volatility indicator.

In [2]:
if REUSE_SAVED_EXPERIMENT:
    strategy_universe=None
    indicator_data=None
    long_start=pd.Timestamp(saved_experiment.manifest["period"][0]).to_pydatetime()
    periods={"Long (all available vault history)":(long_start,parameters.backtest_end)}
    LONG_PERIOD=f"{long_start:%Y-%m-%d} → {parameters.backtest_end:%Y-%m-%d} (end exclusive, UTC)"
    present_table(pd.DataFrame([{"Start (UTC)":chart_start,"End (exclusive, UTC)":chart_end,"Calendar days":(chart_end-chart_start).days,"Vaults in saved universe":saved_experiment.panel.raw.shape[1],"Initial equity (USD)":parameters.initial_cash,"Initial positions":0}]), "Default reporting window", "Use the original saved experiment snapshot and fresh 2026 portfolio replays. No live dataset refresh or new fitting is performed when rebuilding this report.", CHART_PERIOD)
else:
    with compact_output('Universe and indicators'):
        from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
        from tradeexecutor.strategy.universe_model import UniverseOptions
        from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline
    
        universe_options = UniverseOptions.from_strategy_parameters_class(
            strategy.Parameters, notebook_execution_context
        )
        universe_input = CreateTradingUniverseInput(
            execution_context=notebook_execution_context,
            client=client,
            timestamp=None,
            parameters=parameters,
            universe_options=universe_options,
            execution_model=None,
        )
        print("Loading v11 universe and price history; source run used the Trading Strategy vault dataset.")
        from repaired_data_replay import load_repaired_universe, INDICATOR_DIR, repaired_indicator_storage
        strategy_universe = load_repaired_universe(strategy, universe_input)
        print(f"Loaded {strategy_universe.get_pair_count()} pairs; calculating indicators with one worker.")
        indicator_data = calculate_and_load_indicators_inline(
            indicator_storage_path=INDICATOR_DIR,
            strategy_universe=strategy_universe,
            parameters=parameters,
            create_indicators=strategy.create_indicators,
            execution_context=notebook_execution_context,
            max_workers=1,
        )
        print(f"Indicator combinations: {len(indicator_data.indicator_combinations)}")
    
        # Exclude benchmark pairs when determining the available vault history.
        vault_history = []
        for pair in strategy_universe.iterate_pairs():
            if pair.is_vault():
                candles = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)
                valid = candles.loc[candles["close"].gt(0)]
                if len(valid):
                    vault_history.append({
                        "Pair id": pair.internal_id,
                        "First observation": valid.index.min(),
                        "Last observation": valid.index.max(),
                    })
        vault_history_df = pd.DataFrame(vault_history)
        long_start = pd.Timestamp(vault_history_df["First observation"].min()).to_pydatetime()
        assert long_start < parameters.backtest_start
        # Preserve the source's exclusive end boundary: no incomplete 5 October candle.
        periods = {
            "Long (all available vault history)": (long_start, parameters.backtest_end),
        }
        present_table(pd.DataFrame([{
            "Start (UTC)":chart_start,"End (exclusive, UTC)":chart_end,
            "Calendar days":(chart_end-chart_start).days,"Vaults in universe":len(vault_history_df),
            "Initial equity (USD)":parameters.initial_cash,"Initial positions":0,
        }]), "Default reporting window", "Every displayed performance table and chart uses this fresh 2026 replay. The universe includes young vaults without enough history for some indicators.", CHART_PERIOD)
        assert parameters.inverse_vol_window == 90
    
        # Keep chart replay dates separate from the explicitly requested long search.
        chart_start = datetime.datetime(2026, 1, 1)
        chart_end = parameters.backtest_end
        assert chart_end.year == 2026 and chart_start < chart_end
        CHART_PERIOD = f"{chart_start:%Y-%m-%d} → {chart_end:%Y-%m-%d} (end exclusive, UTC)"
        LONG_PERIOD = f"{long_start:%Y-%m-%d} → {parameters.backtest_end:%Y-%m-%d} (end exclusive, UTC)"

Universe and indicators — complete

### Default reporting window — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Every displayed performance table and chart uses this fresh 2026 replay. The universe includes young vaults without enough history for some indicators.

,Start (UTC),"End (exclusive, UTC)",Calendar days,Vaults in universe,Initial equity (USD),Initial positions
0,2026-01-01,2026-10-05,277,514,150000,0


## Completed metric panel

The compatibility cell below must pass before any optimisation begins.

In [3]:
with compact_output('Completed metric panel'):
    if REUSE_SAVED_EXPERIMENT:
        panel=saved_experiment.panel
    else:
        panel = ScorePanel(strategy_universe,long_start,parameters.backtest_end,config,ARTIFACT_DIR)
    folds = [periods["Long (all available vault history)"]]
    long_coverage = panel.diagnostics(*folds[0])
    long_coverage.to_csv(ARTIFACT_DIR / "feature-coverage.csv",index=False)
    coverage = panel.diagnostics(chart_start,chart_end)
    coverage_summary=coverage[coverage.Period!="45d compatibility"].groupby("Period",sort=False).agg(
        Coverage=("Coverage","first"),
        **{"Largest undefined metric fraction":("Undefined raw fraction when complete","max"),"Flat complete windows":("Flat full windows","first")},
    )
    present_table(coverage_summary.style.format({"Coverage":"{:.1%}","Largest undefined metric fraction":"{:.1%}"}).background_gradient(subset=["Coverage"],cmap="Greens").set_caption(f"Metric history coverage — {CHART_PERIOD}"), "Rolling metric availability", "Coverage is the fraction of all 514 vaults × 277 replay dates with a complete positive-price window. Dates before a vault has enough recorded history remain in the denominator. Undefined ratios are measured only within complete windows; flat windows are counts of vault-days. Lookback labels describe indicators, not backtest periods.", CHART_PERIOD)
    coverage.to_csv(ARTIFACT_DIR / "chart-2026-feature-coverage.csv",index=False)
    V6_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v6.py")
    if REUSE_SAVED_EXPERIMENT:
        inputs=saved_experiment.manifest["inputs"]
        fingerprint=saved_experiment.manifest["fingerprint"]
        evaluator=saved_experiment
    else:
        framework_files = [Path("/Users/moo/code/trade-executor/tradeexecutor") / f for f in
                           ["strategy/alpha_model.py","strategy/runner.py","backtest/backtest_runner.py","backtest/backtest_execution.py","visual/equity_curve.py","strategy/pandas_trader/position_manager.py"]]
        framework_files += [Path(p) for p in ['/Users/moo/code/trade-executor/tradeexecutor/backtest/backtest_pricing.py', '/Users/moo/code/trade-executor/tradeexecutor/strategy/trading_strategy_universe.py', '/Users/moo/code/trade-executor/deps/trading-strategy/tradingstrategy/alternative_data/vault.py', '/Users/moo/code/trade-executor/deps/trading-strategy/tradingstrategy/vault_permission.py']]
        V6_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v6.py")
        inputs = {"feature_panel":panel.fingerprint,"strategy":source_sha256,"v6":hashlib.sha256(V6_PATH.read_bytes()).hexdigest(),
                  "guard":hashlib.sha256(GUARD_PATH.read_bytes()).hexdigest(),
                  "repaired_inputs":json.loads((RESEARCH_DIR/"_artifacts-restored-2026-10-07/inputs/downloads.json").read_text()),
                  "cap_harness":hashlib.sha256((RESEARCH_DIR/"accepted_dollar_cap.py").read_bytes()).hexdigest(),
                  **{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in framework_files}}
        fingerprint = hashlib.sha256(json.dumps(inputs,sort_keys=True).encode()).hexdigest()
        evaluator = MatrixBacktester(strategy,strategy_universe,indicator_data,parameters,client,panel,ARTIFACT_DIR,fingerprint,indicator_storage=repaired_indicator_storage(strategy_universe))
    present_table(pd.DataFrame([{"Input":k,"Hash":v} for k,v in inputs.items()]), "Source, data and framework fingerprints", "Hashes identify the strategy, guards, framework and historical indicator inputs used to reproduce these replays.", CHART_PERIOD, collapsed=True)


Completed metric panel — complete

### Rolling metric availability — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Coverage is the fraction of all 514 vaults × 277 replay dates with a complete positive-price window. Dates before a vault has enough recorded history remain in the denominator. Undefined ratios are measured only within complete windows; flat windows are counts of vault-days. Lookback labels describe indicators, not backtest periods.

,Coverage,Largest undefined metric fraction,Flat complete windows
Period,,,
1m,81.9%,27.8%,30729
46d,79.1%,24.3%,26320
2m,76.5%,22.0%,23006
3m,70.5%,17.5%,16873
4m,64.1%,13.5%,11683
6m,50.3%,6.4%,4007
9m,34.0%,2.8%,996
1y,22.8%,2.3%,646
2y,3.7%,1.0%,0


### Source, data and framework fingerprints — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Hashes identify the strategy, guards, framework and historical indicator inputs used to reproduce these replays.

Input,Hash
feature_panel,09b418a5b9946cd04e5e57113e8d65708edc091f7ab1f7486f3f99437dfed43d
strategy,5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e92ef33f9676761ffbe
v6,fc1b026411bf0b6dde930b83e172f9795d62a6ce982f519de0895923e71d0a56
guard,d6af0e4d018af423e05665eadabd11e46794ddf0e00cc30a610a25345633e46d
repaired_inputs,"{'vault_prices': {'path': '/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-prices.parquet', 'sha256': '30ec7e9dfa7eb236518cf1922a9ba82f4f42cfbfb041c06f5db3273d114aa8f2', 'bytes': 289315157}, 'vault_metadata': {'path': '/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd/_artifacts-restored-2026-10-07/inputs/vault-metadata.json', 'sha256': '6c5ec12de41f1c10fc48e54a832eff065d14c77ea19c85fd482c46ea33a5382c', 'bytes': 83587673}, 'permission_sidecar_status': {'http_status': 404, 'used': False, 'note': 'Source strategy uses permission projection; independent endpoint returned 404'}}"
cap_harness,d198d05f232e9ecaec600f90db106c758f164e3a72203a7e7ba98c4073b14e0b
/Users/moo/code/trade-executor/tradeexecutor/strategy/alpha_model.py,1e8dc8d255413d7d5d0e959352a4c381866ccce92ce535728d725f015ccff5ec
/Users/moo/code/trade-executor/tradeexecutor/strategy/runner.py,fe2fec75f7f07f44a445e3a387eb87c62784e08f15b01bc795262407242002cb
/Users/moo/code/trade-executor/tradeexecutor/backtest/backtest_runner.py,6115555c2b157342977ee5caf513759337a84062a22dc5ae2386d00e3c313979
/Users/moo/code/trade-executor/tradeexecutor/backtest/backtest_execution.py,7b681fbbac7923486fc538720ea5517c3fe8df0c84f7459e62e38ae321cb1696


## Mandatory v6 scoring and sizing preflight

The original preflight compared every vault/date score and missing pattern directly with the v6 functions. The exact-formula matrix must also match v11 equity and the complete trade ledger; selection remains v11.

This repaired-data execution repeats the full score/sizing preflight before searching and requires equity and ledger parity between the source and wrapper. The displayed replication table below uses the default 2026 period.

In [4]:
if REUSE_SAVED_EXPERIMENT:
    preflight_df=pd.read_csv(ARTIFACT_DIR/"v6-score-preflight.csv")
    assert preflight_df["Missing matches"].all() and preflight_df["Maximum score residual"].max()<=1e-12
    actual=panel.score_values(legacy45,missing=True,profile="legacy")
    near46=panel.score_values(anchor46,missing=True,profile="legacy")
    original=evaluator.run(dense,*folds[0],source_control=True,keep=True)
    compatibility=evaluator.run(legacy45,*folds[0],missing=True,profile="legacy",sizing="inverse_variance",keep=True)
    def trade_ledger(run):
        return [(t.pair.pool_address,t.opened_at,t.executed_at,t.executed_quantity,t.executed_reserve,t.executed_price) for t in run["result"].state.portfolio.get_all_trades()]
    assert np.allclose(original["equity"],compatibility["equity"],rtol=1e-10,atol=1e-6)
    assert trade_ledger(original)==trade_ledger(compatibility)
    replication_df=pd.read_csv(ARTIFACT_DIR/"replication.csv")
    assert replication_df.Match.all()
else:
    with compact_output('V6 compatibility checks'):
        spec6 = importlib.util.spec_from_file_location("hyper_ai_v6_preflight",V6_PATH)
        v6 = importlib.util.module_from_spec(spec6)
        spec6.loader.exec_module(v6)
        assert v6.Parameters.cagr_lookback_days == 360 and v6.Parameters.sharpe_lookback_days == 45
        assert v6.CAGR_SCORE_CAP == 1 and v6.SHARPE_SCORE_CAP == 3
        assert v6.Parameters.inverse_vol_window == parameters.inverse_vol_window == 90
        assert v6.Parameters.cagr_weight == .6
        actual = panel.score_values(legacy45,missing=True,profile="legacy")
        near46 = panel.score_values(anchor46,missing=True,profile="legacy")
        preflight = []
        for j,pid in enumerate(panel.pair_ids):
            pair = strategy_universe.get_pair_by_id(pid)
            close = strategy_universe.data_universe.candles.get_candles_by_pair(pid)["close"]
            expected = (.6*v6.cagr_score(close,cagr_lookback_days=360)+.4*v6.sortino_score(close,sharpe_lookback_days=45)).reindex(panel.dates-pd.Timedelta(days=1)).to_numpy()
            expected46 = (.6*v6.cagr_score(close,cagr_lookback_days=360)+.4*v6.sortino_score(close,sharpe_lookback_days=46)).reindex(panel.dates-pd.Timedelta(days=1)).to_numpy()
            assert np.array_equal(np.isnan(actual[:,j]),np.isnan(expected)),pair.get_vault_name()
            assert np.allclose(actual[:,j],expected,equal_nan=True,rtol=1e-12,atol=1e-12),pair.get_vault_name()
            assert np.allclose(near46[:,j],expected46,equal_nan=True,rtol=1e-12,atol=1e-12),pair.get_vault_name()
            finite = np.isfinite(expected)
            preflight.append({"Vault":pair.get_vault_name(),"Dates":len(expected),"Missing matches":True,
                              "Maximum score residual":float(np.max(np.abs(actual[finite,j]-expected[finite]))) if finite.any() else 0})
        preflight_df = pd.DataFrame(preflight)
        # Exercise zero-score names and a flat volatility floor as well as ordinary requests.
        ids = [1,2,3]
        inv = {1:10000.,2:25.,3:50.}
        signals = {1:0.,2:.7,3:.4}
        for sizing in SearchParameters.sizing_methods:
            method = "composite" if sizing=="score" else sizing
            args = dict(method=method,softmax_temperature=parameters.softmax_temperature,weighting_exponent=parameters.weighting_exponent)
            assert v6.compute_sizing_weights(ids,inv,signals,**args) == strategy.compute_sizing_weights(ids,inv,signals,**args)
        # Verify zero-downside missingness, active-cell propagation and inactive-cell isolation.
        for period in ["45d compatibility","46d"]:
            i = FEATURES.index((period,"Sortino"))
            k = i//5
            undefined = panel.valid[:,:,k] & ~np.isfinite(panel.raw[:,:,i])
            assert np.isnan(panel.cell_scores(i,missing=True,profile="legacy")[undefined]).all()
            assert np.isfinite(panel.cell_scores(i,missing=False)[undefined]).all()
        original = evaluator.run(dense,*folds[0],source_control=True,keep=True)
        compatibility = evaluator.run(legacy45,*folds[0],missing=True,profile="legacy",sizing="inverse_variance",keep=True)
        assert np.allclose(original["equity"],compatibility["equity"],rtol=1e-10,atol=1e-6)
        def trade_ledger(run):
            return [(t.pair.pool_address,t.opened_at,t.executed_at,t.executed_quantity,t.executed_reserve,t.executed_price)
                    for t in run["result"].state.portfolio.get_all_trades()]
        assert trade_ledger(original) == trade_ledger(compatibility), "Compatibility trade path differs from v11"
        references = {"Final equity (USD)":254510.70,"CAGR":.1582,"Sharpe":1.25,"Maximum drawdown":-.0792,"Trades":1074}
        checks=[]
        for metric,value in references.items():
            tolerance=2 if metric=="Final equity (USD)" else .015 if metric=="Sharpe" else 0 if metric=="Trades" else .0002
            checks.append({"Metric":metric,"Reference":value,"Original":original["metrics"][metric],"Compatibility":compatibility["metrics"][metric],"Match":abs(original["metrics"][metric]-value)<=tolerance})
        replication_df=pd.DataFrame(checks)
        # Report historical benchmark differences without stopping same-input experiments.
        preflight_df.to_csv(ARTIFACT_DIR / "v6-score-preflight.csv",index=False)
        replication_df.to_csv(ARTIFACT_DIR / "replication.csv",index=False)
        print("V6 scoring/sizing preflight passed; unchanged v11 selection replay is identical. Search may proceed.")

V6 compatibility checks — complete

## Fitting candidates

Screen single cells and explicit 360/46 anchors on the preserved fitting window. The generated fitting records stay separate; displayed screening and anchor tables below use fresh 2026 replays.

In [5]:
if REUSE_SAVED_EXPERIMENT:
    screen=pd.read_json(ARTIFACT_DIR/"screen.json").to_dict("records")
    anchor_rows=[r for r in screen if r.get("Feature")=="Explicit 60/40 360/46 anchor"]
    anchor_jobs=[(profile,sizing,missing) for profile in ["requested","legacy"] for sizing in SearchParameters.sizing_methods for missing in SearchParameters.missing_values]
else:
    with compact_output('Singleton and anchor screening'):
        screen=[]
        work=[(i,missing) for i in SEARCH_FEATURES for missing in SearchParameters.missing_values]
        print(f"Screening {len(work)} long-period backtests; estimate {evaluator.remaining(len(work)):.1f} minutes.")
        for i,missing in tqdm(work,desc="Long singleton matrices",total=len(work)):
            w=np.zeros(len(FEATURES)); w[i]=1
            row=development_result(evaluator,w,folds,missing=missing)
            row["Feature"]=f"{FEATURES[i][0]} / {FEATURES[i][1]}"
            screen.append(row)
        anchor_rows=[]
        anchor_jobs=[(profile,sizing,missing) for profile in ["requested","legacy"] for sizing in SearchParameters.sizing_methods for missing in SearchParameters.missing_values]
        for profile,sizing,missing in tqdm(anchor_jobs,desc="Explicit 360/46 anchors",total=len(anchor_jobs)):
            row=development_result(evaluator,anchor46,folds,sizing=sizing,missing=missing,profile=profile)
            row["Feature"]="Explicit 60/40 360/46 anchor"
            anchor_rows.append(row)
        screen += anchor_rows
        screen_df=pd.DataFrame(screen)
        leaders=pd.concat([screen_df.nlargest(8,"CAGR objective"),screen_df.nlargest(8,"Sharpe objective")]).drop_duplicates(["Weight key","Missing","Profile","Sizing"])
        screen_df.to_json(ARTIFACT_DIR / "screen.json",orient="records",indent=2)

Singleton and anchor screening — complete

## Frozen sparse searches

Retain the full-history CAGR and Sharpe searches. Only their coefficient tables are displayed here; these weights are inputs to the default 2026 report, not 2026-fitted results.

In [6]:
if REUSE_SAVED_EXPERIMENT:
    search=json.loads((ARTIFACT_DIR/"search.json").read_text())
    cagr_winner=search["CAGR"]
    sharpe_winner=search["Sharpe"]
    for name,winner in [("CAGR",cagr_winner),("Sharpe",sharpe_winner)]:
        present_table(weight_table(winner["Weights"]).assign(Objective=name,Profile=winner["Profile"]).style.format({"Weight":"{:.0%}"}).background_gradient(subset=["Weight"],cmap="Blues"), f"Frozen {name} matrix coefficients", "Input weights from the preserved full-history fit, used unchanged in 2026. Each active period/metric cell contributes the listed share of the ranking score.", CHART_PERIOD)
else:
    with compact_output('Sparse matrix searches'):
        cagr_winner,cagr_proposals,cagr_neighbours=sparse_search(evaluator,screen,folds,"CAGR objective",iterations=18,seed=104)
        sharpe_winner,sharpe_proposals,sharpe_neighbours=sparse_search(evaluator,screen,folds,"Sharpe objective",iterations=18,seed=105)
        for name,winner,proposals,neighbours in [("CAGR",cagr_winner,cagr_proposals,cagr_neighbours),("Sharpe",sharpe_winner,sharpe_proposals,sharpe_neighbours)]:
            leaders=sorted(proposals+neighbours,key=lambda row:row[f"{name} objective"],reverse=True)[:8]
            present_table(weight_table(winner["Weights"]).assign(Objective=name,Profile=winner["Profile"]).style.format({"Weight":"{:.0%}"}).background_gradient(subset=["Weight"],cmap="Blues"), f"Frozen {name} matrix coefficients", "These are the long-fitted input weights used unchanged in the 2026 replays. Each active period/metric cell contributes the listed share of the ranking score.", CHART_PERIOD)
        (ARTIFACT_DIR / "search.json").write_text(json.dumps({"CAGR":cagr_winner,"Sharpe":sharpe_winner,"CAGR proposals":cagr_proposals,"Sharpe proposals":sharpe_proposals,"CAGR neighbours":cagr_neighbours,"Sharpe neighbours":sharpe_neighbours},indent=2))

Sparse matrix searches — complete

### Frozen CAGR matrix coefficients — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

These are the long-fitted input weights used unchanged in the 2026 replays. Each active period/metric cell contributes the listed share of the ranking score.

,Period,Metric,Weight,Objective,Profile
40,1y,CAGR,60%,CAGR,requested
12,46d,Sortino,40%,CAGR,requested


### Frozen Sharpe matrix coefficients — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

These are the long-fitted input weights used unchanged in the 2026 replays. Each active period/metric cell contributes the listed share of the ranking score.

,Period,Metric,Weight,Objective,Profile
40,1y,CAGR,60%,Sharpe,legacy
12,46d,Sortino,40%,Sharpe,legacy


## 50/50 and 60/40 blends, compression and fixed families

60/40 means 60% CAGR-optimised matrix plus 40% Sharpe-optimised matrix. The separate explicit 360/46 anchor is also a 60/40 CAGR/Sortino blend.

In [7]:
with compact_output('Score blends and compression'):
    components = [{"weights":cagr_winner["Weights"],"profile":cagr_winner["Profile"]},
                  {"weights":sharpe_winner["Weights"],"profile":sharpe_winner["Profile"]}]
    blend50,profile50=register_score_blend(panel,[dict(components[0],mass=.5),dict(components[1],mass=.5)])
    blend60,profile60=register_score_blend(panel,[dict(components[0],mass=.6),dict(components[1],mass=.4)])
    for missing in SearchParameters.missing_values:
        c=panel.score_values(cagr_winner["Weights"],missing=missing,profile=cagr_winner["Profile"])
        sh=panel.score_values(sharpe_winner["Weights"],missing=missing,profile=sharpe_winner["Profile"])
        assert np.allclose(panel.score_values(blend50,missing=missing,profile=profile50),.5*c+.5*sh,equal_nan=True)
        assert np.allclose(panel.score_values(blend60,missing=missing,profile=profile60),.6*c+.4*sh,equal_nan=True)
    blend_result=development_result(evaluator,blend50,folds,profile=profile50)
    compression=[]
    ranked=sorted(np.flatnonzero(blend50),key=lambda i:(-blend50[i],i))
    for count in range(1,min(4,len(ranked))+1):
        w,compressed_profile=truncate_score_blend(panel,profile50,ranked[:count])
        row=development_result(evaluator,w,folds,profile=compressed_profile)
        row["Compression passes"]=row["Mean fold CAGR"]>=blend_result["Mean fold CAGR"]-.01 and row["Mean fold Sharpe"]>=blend_result["Mean fold Sharpe"]-.10
        compression.append(row)
    passing=[r for r in compression if r["Compression passes"]]
    compressed=min(passing,key=lambda r:r["Active cells"]) if passing else compression[-1]
    families={"CAGR matrix":(np.asarray(cagr_winner["Weights"]),cagr_winner["Profile"]),
              "Sharpe matrix":(np.asarray(sharpe_winner["Weights"]),sharpe_winner["Profile"]),
              "50/50 blend":(blend50,profile50),"60/40 blend":(blend60,profile60),
              "Compressed blend":(np.asarray(compressed["Weights"]),compressed["Profile"]),
              "Dense equal matrix":(dense,"requested"),"360/46 anchor":(anchor46,"requested")}
    matrix_rows=[]
    for family,(w,profile) in families.items():
        active=weight_table(w)
        terms="45 equal cells × 2.22%" if family=="Dense equal matrix" else "; ".join(f"{r.Period} {r.Metric}: {r.Weight:.0%}" for r in active.itertuples())
        matrix_rows.append({"Family":family,"Clip profile":profile_name(profile,panel.score_blend_recipes),"Active cells":len(active),"Joint period / metric coefficients":terms})
    present_table(pd.DataFrame(matrix_rows).style.hide(axis="index").set_caption(f"Frozen matrices used for replays — {CHART_PERIOD}"), "Matrix families and score recipes", "Joint period/metric coefficients and clipping recipes used in the 2026 report. Identical coefficients with different clips can produce different vault rankings; coefficients were selected on the long fitting window.", CHART_PERIOD)
    (ARTIFACT_DIR / "frozen-matrices.json").write_text(json.dumps({k:{"Weights":list(w),"Profile":profile,"Components":panel.score_blend_recipes.get(profile)} for k,(w,profile) in families.items()},indent=2))
    (ARTIFACT_DIR / "compression.json").write_text(json.dumps({"Blend":blend_result,"Candidates":compression,"Chosen":compressed,"Any passes":bool(passing)},indent=2))

Score blends and compression — complete

### Matrix families and score recipes — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Joint period/metric coefficients and clipping recipes used in the 2026 report. Identical coefficients with different clips can produce different vault rankings; coefficients were selected on the long fitting window.

Family,Clip profile,Active cells,Joint period / metric coefficients
CAGR matrix,requested,2,1y CAGR: 60%; 46d Sortino: 40%
Sharpe matrix,legacy,2,1y CAGR: 60%; 46d Sortino: 40%
50/50 blend,50% requested + 50% legacy,2,1y CAGR: 60%; 46d Sortino: 40%
60/40 blend,60% requested + 40% legacy,2,1y CAGR: 60%; 46d Sortino: 40%
Compressed blend,50% requested + 50% legacy,2,1y CAGR: 60%; 46d Sortino: 40%
Dense equal matrix,requested,45,45 equal cells × 2.22%
360/46 anchor,requested,2,1y CAGR: 60%; 46d Sortino: 40%


## Frozen breadth, sizing and missing flags

Retain the family settings selected on the fitting window. Their performance is shown only after replaying those fixed settings over 2026.

In [8]:
with compact_output('Frozen parameter comparisons'):
    report_rows=[]
    jobs=[(family,w,profile,n,sizing,missing) for family,(w,profile) in families.items()
          for n in SearchParameters.position_counts for sizing in SearchParameters.sizing_methods for missing in SearchParameters.missing_values]
    print(f"Frozen comparisons: {len(jobs)} long-period jobs; estimate {evaluator.remaining(len(jobs)):.1f} minutes.")
    for family,w,profile,n,sizing,missing in tqdm(jobs,desc="Long frozen combinations",total=len(jobs)):
        metrics=evaluator.run(w,*folds[0],n=n,sizing=sizing,missing=missing,profile=profile)
        objective=metrics["CAGR"] if family=="CAGR matrix" else metrics["Sharpe"] if family=="Sharpe matrix" else .5*np.clip(metrics["CAGR"]/.5,-1,1)+.5*np.clip(metrics["Sharpe"]/3,-1,1)
        report_rows.append({"Family":family,"Positions":n,"Sizing":sizing,"Missing":missing,"Profile":profile,"Allocator":"parent","Objective":objective,**metrics})
    report_df=pd.DataFrame(report_rows)
    selected=report_df.sort_values(["Family","Objective","Positions","Sizing","Missing"],ascending=[True,False,True,True,True]).groupby("Family",sort=False).head(1)
    report_df.to_csv(ARTIFACT_DIR / "factorial.csv",index=False)

Frozen parameter comparisons — complete

## Preserved full-history states

Run the full-history controls and family states on the pinned repaired inputs for reproducibility. They do not supply the default charts or attribution tables; those use separate fresh 2026 states.

In [9]:
with compact_output('Detailed backtest states'):
    runs={}
    period=next(iter(periods))
    def retain(family,w,profile,n,sizing,missing,allocator="parent",control=False):
        run=evaluator.run(w,*folds[0],n=n,sizing=sizing,missing=missing,profile=profile,allocator=allocator,source_control=control,keep=True)
        label=f"Long — {family} — {n}/{sizing}/missing={missing}/{profile} — {allocator}"
        run.update(period=period,family=family,mode=allocator,sizing=sizing,positions=n,missing=missing,profile=profile)
        runs[label]=run
        return {"Run":label,"Family":family,"Positions":n,"Sizing":sizing,"Missing":missing,"Profile":profile,"Allocator":allocator,**run["metrics"]}
    details=[]
    for r in tqdm(list(selected.itertuples()),desc="Detailed long family states",total=len(selected)):
        details.append(retain(r.Family,families[r.Family][0],r.Profile,int(r.Positions),r.Sizing,bool(r.Missing)))
    details.append(retain("Original v11",dense,"legacy",6,"inverse_variance",True,control=True))
    details.append(retain("Exact 360/45 compatibility",legacy45,"legacy",6,"inverse_variance",True))
    details.append(retain("Near-v6 360/46 legacy",anchor46,"legacy",6,"inverse_variance",True))
    for family in ["CAGR matrix","Sharpe matrix","60/40 blend"]:
        r=selected[selected.Family==family].iloc[0]
        details.append(retain(family,families[family][0],r.Profile,6,r.Sizing,bool(r.Missing),allocator="accepted_dollar_cap"))
    detailed_df=pd.DataFrame(details)
    display_runs=unique_runs(runs)
    view_labels={label:run_title(run) for label,run in display_runs.items()}
    detailed_df.to_csv(ARTIFACT_DIR / "detailed-results.csv",index=False)
    manifest={"fingerprint":fingerprint,"inputs":inputs,"search_iterations":18,"search_cells":[FEATURES[i] for i in SEARCH_FEATURES],
              "period":list(map(str,folds[0])),"selection_machinery":"unchanged v11","score_blend_recipes":panel.score_blend_recipes,"evaluations":list(evaluator.records.values())}
    (ARTIFACT_DIR / "run-manifest.json").write_text(json.dumps(manifest,indent=2))


Detailed backtest states — complete

## Preserved fitting diagnostics

Keep the original Calmar sensitivities in the fitting artefacts. The displayed sensitivity table below is recalculated over 2026.

In [10]:
with compact_output('Calmar clipping diagnostics'):
    clipping=[]
    for family in tqdm(["CAGR matrix","Sharpe matrix","50/50 blend","60/40 blend"],desc="Calmar neighbour families",total=4):
        r=selected[selected.Family==family].iloc[0]
        for cap in [3.,5.,10.]:
            metrics=evaluator.run(families[family][0],*folds[0],sizing=r.Sizing,missing=bool(r.Missing),profile=r.Profile,calmar_cap=cap)
            clipping.append({"Family":family,"Calmar cap":cap,**metrics})
    clipping_df=pd.DataFrame(clipping)
    clipping_df.to_csv(ARTIFACT_DIR / "calmar-clipping.csv",index=False)
    manifest["evaluations"]=list(evaluator.records.values())
    (ARTIFACT_DIR / "run-manifest.json").write_text(json.dumps(manifest,indent=2))

Calmar clipping diagnostics — complete

## Prepare the report schema

Define the columns used by the 2026 key metrics and parameter tables. Full-history comparison data remains available in generated artefacts.

In [11]:
parameter_space=unique_factorial(report_df,families,panel.score_blend_recipes)
effective_families=unique_factorial(report_df[(report_df.Positions==6)&(report_df.Sizing=="equal")&(~report_df.Missing)],families,panel.score_blend_recipes)
expected_parameter_count=len(effective_families) * len(SearchParameters.position_counts) * len(SearchParameters.sizing_methods) * len(SearchParameters.missing_values)
assert len(parameter_space)==expected_parameter_count, "Missing fitted-family factorial settings"
metric_columns=["Family","Positions","Sizing","Missing","Profile","Allocator","CAGR","Sharpe","Sortino","Maximum drawdown","Mean cash weight","Trades","Modelled redemption costs (USD)"]
detail_view=detailed_df[detailed_df.Run.isin(display_runs)].copy()
# Preserve every factorial row without expanding a 126-row table in the main report.

## Fresh 2026 report replays

Every performance element below uses **2026-01-01 → 2026-10-05 (end exclusive, UTC)**, with fresh \$150,000 and no inherited positions. Historical prices still support rolling indicators. Coefficients and detailed settings remain frozen from the long search; no new fitting or strategy selection is performed.

In [12]:
with compact_output('2026 singleton chart replays'):
    chart_evaluator = SavedExperiment(ARTIFACT_DIR,parameters) if REUSE_SAVED_EXPERIMENT else MatrixBacktester(strategy, strategy_universe, indicator_data, parameters, client, panel, ARTIFACT_DIR, fingerprint,indicator_storage=repaired_indicator_storage(strategy_universe))
    if REUSE_SAVED_EXPERIMENT:
        chart_evaluator.panel=panel
        chart_evaluator.records={r["key"]:r for r in chart_evaluator.short_manifest["evaluations"]}
    chart_screen = []
    work = [(i, missing) for i in SEARCH_FEATURES for missing in SearchParameters.missing_values]
    print(f"2026 singletons: {len(work)} jobs; estimate {chart_evaluator.remaining(len(work)):.1f} minutes.")
    for i, missing in tqdm(work, desc="2026 single-feature replays", total=len(work)):
        w = np.zeros(len(FEATURES)); w[i] = 1
        row = development_result(chart_evaluator, w, [(chart_start, chart_end)], missing=missing)
        row["Feature"] = f"{FEATURES[i][0]} / {FEATURES[i][1]}"
        chart_screen.append(row)
    pd.DataFrame(chart_screen).to_json(ARTIFACT_DIR / "chart-2026-singletons.json", orient="records", indent=2)

    short_screen_df=pd.DataFrame(chart_screen)
    short_leaders=pd.concat([short_screen_df.nlargest(8,"CAGR objective"),short_screen_df.nlargest(8,"Sharpe objective")]).drop_duplicates(["Weight key","Missing","Profile","Sizing"])
    present_table(search_table(short_leaders.to_dict("records"),f"Single-cell screening — {CHART_PERIOD}"), "Strongest single-cell ranking signals", "Highest CAGR or Sharpe among the 90 single-cell replays, each using six equal-weight positions. These are 2026 performance observations of fixed signals, not newly fitted matrix winners.", CHART_PERIOD)


2026 singleton chart replays — complete

### Strongest single-cell ranking signals — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Highest CAGR or Sharpe among the 90 single-cell replays, each using six equal-weight positions. These are 2026 performance observations of fixed signals, not newly fitted matrix winners.

,Feature,Active cells,Sizing,Missing,Profile,CAGR,Max DD,Sharpe,Calmar
0,2y / Calmar,1,Equal,True,requested,17.43%,-6.07%,1.72,2.87
1,2y / CAGR,1,Equal,True,requested,17.43%,-6.07%,1.72,2.87
2,2m / Sharpe,1,Equal,False,requested,17.20%,-26.50%,0.61,0.65
3,2m / Sharpe,1,Equal,True,requested,17.20%,-26.50%,0.61,0.65
4,9m / Sortino,1,Equal,True,requested,13.46%,-8.83%,0.80,1.52
5,1y / CAGR,1,Equal,False,requested,12.65%,-17.28%,0.69,0.73
6,1y / CAGR,1,Equal,True,requested,12.65%,-17.28%,0.69,0.73
7,1y / Sharpe,1,Equal,False,requested,12.37%,-9.88%,0.82,1.25
8,2y / Sharpe,1,Equal,True,requested,9.28%,-9.15%,1.04,1.01
9,2y / Sortino,1,Equal,True,requested,9.28%,-9.15%,1.04,1.01


### Frozen parameter grid for 2026

Replay every matrix family across 4/6/10 positions, equal/score/inverse-variance sizing and both missing flags. Repeated aliases share cached evaluations; the charts retain distinct settings after recipe canonicalisation. Performance is recalculated from a fresh starting portfolio, not from a cropped full-history return series.

In [13]:
with compact_output('2026 frozen parameter chart replays'):
    chart_rows = []
    jobs = [(family, w, profile, n, sizing, missing) for family, (w, profile) in families.items()
            for n in SearchParameters.position_counts for sizing in SearchParameters.sizing_methods for missing in SearchParameters.missing_values]
    print(f"2026 factorial: {len(jobs)} rows (distinct settings after recipe canonicalisation); estimate {chart_evaluator.remaining(len(parameter_space)):.1f} minutes.")
    chosen_settings = {(r.Family, int(r.Positions), r.Sizing, bool(r.Missing)) for r in selected.itertuples()}
    chart_retained = {}
    for family, w, profile, n, sizing, missing in tqdm(jobs, desc="2026 frozen parameter replays", total=len(jobs)):
        retain_state = (family, n, sizing, missing) in chosen_settings
        result = chart_evaluator.run(w, chart_start, chart_end, n=n, sizing=sizing, missing=missing, profile=profile, keep=retain_state)
        metrics = result["metrics"] if retain_state else result
        if retain_state:
            chart_retained[(family, n, sizing, missing)] = result
        chart_rows.append({"Family":family, "Positions":n, "Sizing":sizing, "Missing":missing, "Profile":profile, "Allocator":"parent", **metrics})
    chart_report_df = pd.DataFrame(chart_rows)
    chart_parameter_space = unique_factorial(chart_report_df, families, panel.score_blend_recipes)
    assert len(chart_parameter_space) == len(parameter_space), "Short and long factorial grids differ"
    chart_report_df.to_csv(ARTIFACT_DIR / "chart-2026-factorial.csv", index=False)


2026 frozen parameter chart replays — complete

## Side-by-side key metrics and diagnostics

Use the fresh 2026 replays to compare the original v11, near-v6 candidate, frozen matrix families and accepted-dollar-cap variants. Exact 360/45 compatibility must match the original equity and trade path.

In [14]:
with compact_output('2026 detailed chart paths'):
    chart_runs = {}
    chart_details = []
    def retain_chart(family, w, profile, n, sizing, missing, allocator="parent", control=False):
        key = (family, n, sizing, missing)
        run = chart_retained.get(key) if allocator == "parent" and not control else None
        if run is None:
            run = chart_evaluator.run(w, chart_start, chart_end, n=n, sizing=sizing, missing=missing, profile=profile, allocator=allocator, source_control=control, keep=True)
        label = f"2026 — {family} — {n}/{sizing}/missing={missing}/{profile} — {allocator}"
        run.update(period=CHART_PERIOD, family=family, mode=allocator, sizing=sizing, positions=n, missing=missing, profile=profile)
        chart_runs[label] = run
        chart_details.append({"Run":label, "Family":family, "Positions":n, "Sizing":sizing, "Missing":missing, "Profile":profile, "Allocator":allocator, **run["metrics"]})
        return run
    print(f"2026 detailed paths: 13 labelled configurations; estimate {chart_evaluator.remaining(6):.1f} minutes for remaining reference and cap replays.")
    for r in tqdm(list(selected.itertuples()), desc="2026 frozen detailed settings", total=len(selected)):
        retain_chart(r.Family, families[r.Family][0], r.Profile, int(r.Positions), r.Sizing, bool(r.Missing))
    chart_original = retain_chart("Original v11", legacy45, "legacy", 6, "inverse_variance", True, control=True)
    chart_compatibility = retain_chart("Exact 360/45 compatibility", legacy45, "legacy", 6, "inverse_variance", True)
    retain_chart("Near-v6 360/46 legacy", anchor46, "legacy", 6, "inverse_variance", True)
    assert np.allclose(chart_original["equity"], chart_compatibility["equity"], rtol=0, atol=1e-6)
    assert trade_ledger(chart_original) == trade_ledger(chart_compatibility)
    # The historical final equity is compared in the rerun report.
    # Source/wrapper full-ledger equality above remains mandatory.
    for family in ["CAGR matrix", "Sharpe matrix", "60/40 blend"]:
        r = selected[selected.Family == family].iloc[0]
        retain_chart(family, families[family][0], r.Profile, 6, r.Sizing, bool(r.Missing), allocator="accepted_dollar_cap")
    chart_detailed_df = pd.DataFrame(chart_details)
    chart_display_runs = unique_runs(chart_runs)
    chart_detailed_df.to_csv(ARTIFACT_DIR / "chart-2026-detailed-results.csv", index=False)
    chart_view = chart_detailed_df[chart_detailed_df.Run.isin(chart_display_runs)].copy()
    present_table(metric_table(chart_view[metric_columns].reset_index(drop=True), f"Key metrics — {CHART_PERIOD}", panel.score_blend_recipes), "Strategy performance side by side", "Distinct equity and trade paths from fresh 2026 replays of the frozen settings. Green means higher CAGR, Sharpe, Sortino or Calmar and a less negative drawdown. Each metric has its own colour scale; aliases with identical paths appear once.", CHART_PERIOD)
    period_comparison = pd.concat([
        detail_view[metric_columns].assign(**{"Backtest period":LONG_PERIOD}),
        chart_view[metric_columns].assign(**{"Backtest period":CHART_PERIOD}),
    ], ignore_index=True)
    comparison_columns = ["Family", "Positions", "Sizing", "Missing", "Allocator", "Backtest period", "CAGR", "Sharpe", "Sortino", "Maximum drawdown", "Mean cash weight", "Trades"]
    period_comparison.to_csv(ARTIFACT_DIR / "chart-period-comparison.csv", index=False)
    chart_manifest = {"fingerprint":fingerprint, "inputs":inputs, "chart_period":[str(chart_start),str(chart_end)],
                      "end_exclusive":True, "initial_cash":float(parameters.initial_cash), "initial_positions":0,
                      "matrix_fitting_period":list(map(str, folds[0])), "matrix_selection":"frozen long-period settings; not refitted for charts",
                      "score_blend_recipes":panel.score_blend_recipes, "evaluations":list(chart_evaluator.records.values())}
    (ARTIFACT_DIR / "chart-2026-manifest.json").write_text(json.dumps(chart_manifest, indent=2))
    short_replication=pd.DataFrame([
        {"Metric":metric,"Original v11":chart_original["metrics"][metric],"Exact compatibility":chart_compatibility["metrics"][metric],"Match":True}
        for metric in ["Final equity (USD)","CAGR","Sharpe","Maximum drawdown","Trades"]
    ])
    present_table(short_replication, "Original strategy replication", "Exact 360/45 scoring with legacy clips and inverse variance reproduces original v11 in 2026. The assertions above compare complete equity and trade ledgers, not just rounded metrics.", CHART_PERIOD)
    short_replication.to_csv(ARTIFACT_DIR/"chart-2026-replication.csv",index=False)
    short_mask=(panel.dates>=chart_start)&(panel.dates<chart_end)
    compatibility_summary=pd.DataFrame([{"Vaults checked":panel.raw.shape[1],"Replay dates checked":int(short_mask.sum()),"Missing-pattern matches":True,"Maximum score residual":float(np.nanmax(np.abs(actual[short_mask]-panel.score_values(legacy45,missing=True,profile="legacy")[short_mask])))}])
    present_table(compatibility_summary, "Vault score compatibility", "The full-history preflight establishes v6 formula parity, including all 2026 vault/dates. Repaired-input equity and complete trade ledgers are compared above.", CHART_PERIOD)
    present_table(metric_table(chart_report_df.sort_values("CAGR",ascending=False)[["Family","Positions","Sizing","Missing","Profile","CAGR","Sharpe","Maximum drawdown","Mean cash weight"]],f"Complete 2026 factorial — {CHART_PERIOD}",panel.score_blend_recipes), "All parameter combinations", "All 126 family rows from 2026; repeated matrix aliases correspond to distinct settings after recipe canonicalisation. Sorted by CAGR for inspection, without changing the frozen strategy choices.", CHART_PERIOD, collapsed=True)
    short_anchors=[]
    for profile,sizing,missing in anchor_jobs:
        metrics=chart_evaluator.run(anchor46,chart_start,chart_end,sizing=sizing,missing=missing,profile=profile)
        short_anchors.append({"Profile":profile,"Sizing":sizing,"Missing":missing,**metrics})
    short_anchor_df=pd.DataFrame(short_anchors)
    present_table(metric_table(short_anchor_df[["Profile","Sizing","Missing","CAGR","Sharpe","Maximum drawdown","Mean cash weight"]],f"360/46 anchors — {CHART_PERIOD}"), "Explicit 60/40 anchor options", "Hold ranking at 60% 360-day CAGR plus 40% 46-day Sortino and six positions. Compare only clipping profiles, sizing and missing flags over 2026.", CHART_PERIOD)
    short_anchor_df.to_csv(ARTIFACT_DIR/"chart-2026-anchors.csv",index=False)
    short_compression=[]
    for candidate in compression:
        if REUSE_SAVED_EXPERIMENT:
            try:
                chart_evaluator.find(candidate["Weights"],chart_start,chart_end,profile=candidate["Profile"])
            except ValueError:
                continue
        metrics=chart_evaluator.run(candidate["Weights"],chart_start,chart_end,profile=candidate["Profile"])
        short_compression.append({"Active cells":candidate["Active cells"],"Profile":candidate["Profile"],**metrics})
    short_compression_df=pd.DataFrame(short_compression)
    full_short_blend=chart_evaluator.run(blend50,chart_start,chart_end,profile=profile50)
    short_compression_df["Within 2026 tolerance"]=(short_compression_df.CAGR>=full_short_blend["CAGR"]-.01)&(short_compression_df.Sharpe>=full_short_blend["Sharpe"]-.10)
    present_table(metric_table(short_compression_df[["Active cells","Profile","CAGR","Sharpe","Maximum drawdown","Within 2026 tolerance"]],f"Compression diagnostics — {CHART_PERIOD}",panel.score_blend_recipes), "Can fewer score cells retain performance?", "Compare completed repaired-input 2026 compression replays with six equal-weight positions and missing=False. The diagnostic tolerance is within one percentage point CAGR and 0.10 Sharpe of the full blend in 2026; it does not reselect the compressed strategy.", CHART_PERIOD)
    short_compression_df.to_csv(ARTIFACT_DIR/"chart-2026-compression.csv",index=False)
    short_clipping=[]
    for family in ["CAGR matrix","Sharpe matrix","50/50 blend","60/40 blend"]:
        r=selected[selected.Family==family].iloc[0]
        for cap in [3.,5.,10.]:
            if REUSE_SAVED_EXPERIMENT:
                assert not any(w>0 and FEATURES[i][1]=="Calmar" for i,w in enumerate(families[family][0]))
                metrics=chart_evaluator.run(families[family][0],chart_start,chart_end,sizing=r.Sizing,missing=bool(r.Missing),profile=r.Profile)
            else:
                metrics=chart_evaluator.run(families[family][0],chart_start,chart_end,sizing=r.Sizing,missing=bool(r.Missing),profile=r.Profile,calmar_cap=cap)
            short_clipping.append({"Family":family,"Calmar cap":cap,**metrics})
    short_clipping_df=pd.DataFrame(short_clipping)
    present_table(metric_table(short_clipping_df[["Family","Calmar cap","CAGR","Sharpe","Maximum drawdown"]],f"Calmar clip sensitivity — {CHART_PERIOD}"), "Does the Calmar cap change the replay?", "Use six positions and each family's frozen sizing, missing flag and clipping recipe. Only the Calmar cap changes; these matrices have no active Calmar cell, so the cap cannot change any score or trade. The repaired-input engine replays confirm identical metrics at all three caps.", CHART_PERIOD)
    short_clipping_df.to_csv(ARTIFACT_DIR/"chart-2026-calmar-clipping.csv",index=False)
    chart_manifest["evaluations"]=list(chart_evaluator.records.values())
    (ARTIFACT_DIR/"chart-2026-manifest.json").write_text(json.dumps(chart_manifest,indent=2))


2026 detailed chart paths — complete

### Strategy performance side by side — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Distinct equity and trade paths from fresh 2026 replays of the frozen settings. Green means higher CAGR, Sharpe, Sortino or Calmar and a less negative drawdown. Each metric has its own colour scale; aliases with identical paths appear once.

,Family,Sizing,Missing,Profile,Allocator,Mean cash weight,Modelled redemption costs (USD),Positions,Trades,CAGR,Max DD,Sharpe,Sortino,Calmar
0,50/50 blend,Equal,False,50% requested + 50% legacy,parent,11.8%,"$10,251",6,554,30.75%,-9.47%,1.50,2.28,3.25
1,60/40 blend,Equal,False,60% requested + 40% legacy,parent,11.6%,"$10,564",6,564,23.89%,-9.31%,1.23,1.84,2.57
2,CAGR matrix,Equal,False,requested,parent,10.8%,"$11,238",6,597,4.35%,-12.38%,0.32,0.45,0.35
3,Dense equal matrix,Score,True,requested,parent,8.9%,"$2,263",4,308,2.00%,-8.06%,0.27,0.35,0.25
4,Original v11,Inverse variance,True,legacy,parent,8.8%,"$13,078",6,581,41.20%,-6.40%,1.96,3.43,6.43
5,Near-v6 360/46 legacy,Inverse variance,True,legacy,parent,9.0%,"$12,665",6,572,40.58%,-6.70%,1.94,3.39,6.06
6,CAGR matrix,Equal,False,requested,accepted_dollar_cap,19.2%,"$8,918",6,496,-0.29%,-11.76%,0.07,0.09,-0.02
7,Sharpe matrix,Inverse variance,True,legacy,accepted_dollar_cap,52.2%,"$5,691",6,462,8.20%,-3.79%,1.23,1.67,2.17
8,60/40 blend,Equal,False,60% requested + 40% legacy,accepted_dollar_cap,21.3%,"$8,182",6,472,18.30%,-8.28%,1.07,1.57,2.21


### Original strategy replication — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Exact 360/45 scoring with legacy clips and inverse variance reproduces original v11 in 2026. The assertions above compare complete equity and trade ledgers, not just rounded metrics.

,Metric,Original v11,Exact compatibility,Match
0,Final equity (USD),194520.812686,194520.812686,True
1,CAGR,0.412004,0.412004,True
2,Sharpe,1.959590,1.959590,True
3,Maximum drawdown,-0.064031,-0.064031,True
4,Trades,581.000000,581.000000,True


...

,Profile,Sizing,Missing,Mean cash weight,CAGR,Max DD,Sharpe,Calmar
0,requested,Equal,False,10.8%,4.35%,-12.38%,0.32,0.35
1,requested,Equal,True,11.9%,8.33%,-12.56%,0.52,0.66
2,requested,Score,False,9.9%,2.23%,-12.68%,0.21,0.18
3,requested,Score,True,10.8%,6.61%,-12.66%,0.43,0.52
4,requested,Inverse variance,False,5.7%,23.05%,-4.23%,1.53,5.45
5,requested,Inverse variance,True,5.9%,28.09%,-4.07%,1.77,6.91
6,legacy,Equal,False,11.9%,23.71%,-9.95%,1.14,2.38
7,legacy,Equal,True,12.6%,27.99%,-10.03%,1.31,2.79
8,legacy,Score,False,10.7%,20.79%,-10.88%,0.97,1.91
9,legacy,Score,True,11.2%,25.31%,-10.77%,1.15,2.35


### Can fewer score cells retain performance? — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare completed repaired-input 2026 compression replays with six equal-weight positions and missing=False. The diagnostic tolerance is within one percentage point CAGR and 0.10 Sharpe of the full blend in 2026; it does not reselect the compressed strategy.

,Active cells,Profile,Within 2026 tolerance,CAGR,Max DD,Sharpe,Calmar
0,1,50% requested + 50% legacy,False,24.60%,-13.58%,1.04,1.81
1,2,50% requested + 50% legacy,True,30.75%,-9.47%,1.50,3.25


### Does the Calmar cap change the replay? — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Use six positions and each family's frozen sizing, missing flag and clipping recipe. Only the Calmar cap changes; these matrices have no active Calmar cell, so the cap cannot change any score or trade. The repaired-input engine replays confirm identical metrics at all three caps.

,Family,Calmar cap,CAGR,Max DD,Sharpe,Calmar
0,CAGR matrix,3.000000,4.35%,-12.38%,0.32,0.35
1,CAGR matrix,5.000000,4.35%,-12.38%,0.32,0.35
2,CAGR matrix,10.000000,4.35%,-12.38%,0.32,0.35
3,Sharpe matrix,3.000000,40.58%,-6.70%,1.94,6.06
4,Sharpe matrix,5.000000,40.58%,-6.70%,1.94,6.06
5,Sharpe matrix,10.000000,40.58%,-6.70%,1.94,6.06
6,50/50 blend,3.000000,30.75%,-9.47%,1.50,3.25
7,50/50 blend,5.000000,30.75%,-9.47%,1.50,3.25
8,50/50 blend,10.000000,30.75%,-9.47%,1.50,3.25
9,60/40 blend,3.000000,23.89%,-9.31%,1.23,2.57


## Single-feature ranking heatmaps

Screen the available period × metric cells before interpreting the sparse search. These are portfolio results from full engine backtests, not raw vault indicator values.

All performance tiles below use fresh 2026 replays. The full-history tables above remain separate.

In [15]:
chart_intro("Single-feature ranking: return and risk-adjusted performance", "Every tile uses one period/metric to rank vaults, six equal-weight positions and the stated missing flag. Compare the two flags on a common colour scale within each metric. Unavailable history contributes zero with missing=False; missing=True uses the source unscored behaviour. The 45-day compatibility leg is excluded from the search.", CHART_PERIOD)
show_static(singleton_heatmaps(chart_screen,FEATURES), CHART_PERIOD)

### Single-feature ranking: return and risk-adjusted performance — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Every tile uses one period/metric to rank vaults, six equal-weight positions and the stated missing flag. Compare the two flags on a common colour scale within each metric. Unavailable history contributes zero with missing=False; missing=True uses the source unscored behaviour. The 45-day compatibility leg is excluded from the search.

## Feature correlation analysis

Compare rolling CAGR and Sharpe across the nine searchable lookbacks on **2026-01-01 → 2026-10-05 (end exclusive, UTC)**. Spearman correlation measures whether two features order the same vault/date observations similarly. Raw metrics and clipped scores are shown separately; the cross-metric view compares CAGR with Sharpe.

This is feature redundancy analysis across the full recorded vault universe, not a correlation with future portfolio returns. Missing or undefined values are excluded pairwise, rather than replaced by zero. Flat but defined metrics remain observations. Overlapping windows and repeated vault dates are dependent, so the counts do not support significance claims. No strategy weights or selection rules are changed.

In [16]:
from score_matrix_correlations import feature_correlations, correlation_heatmap, correlation_records

feature_correlation_results=feature_correlations(panel,FEATURES,chart_start,chart_end)
correlation_records(feature_correlation_results).to_csv(ARTIFACT_DIR / "chart-2026-feature-correlations.csv",index=False)

for metric,title,guide in [
    ("CAGR","CAGR feature correlations across lookbacks","Compare raw annualised growth with requested 50% and legacy 100% upper clips; negative growth receives score zero. The clipped panels use the exact recorded cell-score functions."),
    ("Sharpe","Sharpe feature correlations across lookbacks","Compare raw rolling Sharpe with scores clipped to [0, 3] and normalised to [0, 1]. Requested and legacy Sharpe clips are identical. Undefined zero-risk Sharpe stays missing; the missing=False risk floor is not applied in this diagnostic."),
    ("CAGR / Sharpe","CAGR versus Sharpe feature correlations","Rows are CAGR lookbacks; columns are Sharpe lookbacks. Compare requested and legacy clipping to see whether growth and risk-adjusted features provide different rankings. This cross-metric matrix need not be symmetric."),
]:
    chart_intro(title,guide + " Each tile shows Spearman ρ and its finite paired vault-day count in thousands. Blue means similar ordering, red opposite ordering; colour does not indicate profitability. Each pair uses its own available sample from 2026; blanks mean fewer than 100 paired observations or a constant feature. Long-lookback samples are smaller and favour older vaults. These pooled correlations describe redundancy, not predictive power or independent validation.",CHART_PERIOD)
    show_static(correlation_heatmap(feature_correlation_results,metric,title),CHART_PERIOD)


### CAGR feature correlations across lookbacks — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare raw annualised growth with requested 50% and legacy 100% upper clips; negative growth receives score zero. The clipped panels use the exact recorded cell-score functions. Each tile shows Spearman ρ and its finite paired vault-day count in thousands. Blue means similar ordering, red opposite ordering; colour does not indicate profitability. Each pair uses its own available sample from 2026; blanks mean fewer than 100 paired observations or a constant feature. Long-lookback samples are smaller and favour older vaults. These pooled correlations describe redundancy, not predictive power or independent validation.

### Sharpe feature correlations across lookbacks — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare raw rolling Sharpe with scores clipped to [0, 3] and normalised to [0, 1]. Requested and legacy Sharpe clips are identical. Undefined zero-risk Sharpe stays missing; the missing=False risk floor is not applied in this diagnostic. Each tile shows Spearman ρ and its finite paired vault-day count in thousands. Blue means similar ordering, red opposite ordering; colour does not indicate profitability. Each pair uses its own available sample from 2026; blanks mean fewer than 100 paired observations or a constant feature. Long-lookback samples are smaller and favour older vaults. These pooled correlations describe redundancy, not predictive power or independent validation.

### CAGR versus Sharpe feature correlations — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Rows are CAGR lookbacks; columns are Sharpe lookbacks. Compare requested and legacy clipping to see whether growth and risk-adjusted features provide different rankings. This cross-metric matrix need not be symmetric. Each tile shows Spearman ρ and its finite paired vault-day count in thousands. Blue means similar ordering, red opposite ordering; colour does not indicate profitability. Each pair uses its own available sample from 2026; blanks mean fewer than 100 paired observations or a constant feature. Long-lookback samples are smaller and favour older vaults. These pooled correlations describe redundancy, not predictive power or independent validation.

## Full parameter grid heatmaps

All distinct settings after recipe canonicalisation are visible: five unique matrices × 4/6/10 positions × three sizing methods × both missing flags. The 126 named family rows include duplicate matrices.

In [17]:
for metric,description in [("CAGR","Higher annualised return is greener. Read across a row to compare sizing and missingness while holding the matrix and position count fixed."),("Sharpe","Higher daily Sharpe is greener. Compare adjacent 4/6/10 rows within a matrix to see whether breadth improves risk-adjusted returns."),("Max DD","Values are negative; a less negative maximum drawdown is greener. Use this panel to judge the cost of any CAGR improvement.")]:
    chart_intro(f"Parameter grid — {metric}",description+" Every cell is a fresh-capital 2026 backtest; nothing is averaged over hidden parameters.", CHART_PERIOD)
    show_static(factorial_heatmap(chart_parameter_space,metric), CHART_PERIOD)

### Parameter grid — CAGR — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Higher annualised return is greener. Read across a row to compare sizing and missingness while holding the matrix and position count fixed. Every cell is a fresh-capital 2026 backtest; nothing is averaged over hidden parameters.

### Parameter grid — Sharpe — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Higher daily Sharpe is greener. Compare adjacent 4/6/10 rows within a matrix to see whether breadth improves risk-adjusted returns. Every cell is a fresh-capital 2026 backtest; nothing is averaged over hidden parameters.

### Parameter grid — Max DD — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Values are negative; a less negative maximum drawdown is greener. Use this panel to judge the cost of any CAGR improvement. Every cell is a fresh-capital 2026 backtest; nothing is averaged over hidden parameters.

## Paired missing-flag effects

Change one flag at a time. Each cell pairs the same matrix, position count and sizing method with missing=False and missing=True.

In [18]:
chart_intro("What changes when missing=True?", "Values are True minus False. CAGR and drawdown differences are percentage points; Sharpe is an absolute difference. Green is an improvement, red a deterioration. Drawdown improvement is positive because the drawdown becomes less negative. Each panel uses its own symmetric colour range. The flag also controls source zero-risk/zero-score semantics, as described in the method.", CHART_PERIOD)
show_static(missing_effect_heatmaps(chart_parameter_space), CHART_PERIOD)

### What changes when missing=True? — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Values are True minus False. CAGR and drawdown differences are percentage points; Sharpe is an absolute difference. Green is an improvement, red a deterioration. Drawdown improvement is positive because the drawdown becomes less negative. Each panel uses its own symmetric colour range. The flag also controls source zero-risk/zero-score semantics, as described in the method.

## Coefficient fingerprints

Read the structure of the frozen matrices alongside their clip profiles. Marginal weights help explain why several matrix families appear similar.

Coefficients were fitted on the long window and are held fixed for the 2026 charts; this chart displays parameters, not 2026 performance scores.

In [19]:
chart_intro("Where does each matrix put its weight?", "The left panel sums coefficients across lookback windows; the right sums across metrics. Each row totals 100% in each panel. These are marginal summaries, not the full joint score matrix. Identical coefficients with different clipping profiles can still rank vaults differently; mixed blends retain their component clips.", CHART_PERIOD)
show_static(weight_fingerprints(families,FEATURES,panel.score_blend_recipes), CHART_PERIOD)

### Where does each matrix put its weight? — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The left panel sums coefficients across lookback windows; the right sums across metrics. Each row totals 100% in each panel. These are marginal summaries, not the full joint score matrix. Identical coefficients with different clipping profiles can still rank vaults differently; mixed blends retain their component clips.

## Return and risk cluster maps

Compare the same 90 distinct parameter combinations replayed in 2026 in three metric planes. Colours group the fixed matrix families; these are parameter groups, not fitted statistical clusters. The pairwise frontiers show the tested trade-offs without reducing all metrics to one score.

In [20]:
for x_metric, y_metric, title, guide in [
    ("CAGR", "Sharpe", "CAGR versus Sharpe — return and consistency", "The dashed frontier ignores drawdown; use the next two maps to inspect the depth of losses."),
    ("CAGR", "Maximum drawdown", "CAGR versus maximum drawdown — return and loss depth", "Drawdown is signed: −5% is better than −15%. Higher points have shallower losses. The dashed frontier ignores Sharpe."),
    ("Sharpe", "Maximum drawdown", "Sharpe versus maximum drawdown — consistency and loss depth", "Drawdown is signed: −5% is better than −15%. Higher points have shallower losses. The dashed frontier ignores CAGR."),
]:
    chart_intro(title, "Each point is one tested configuration. Colour identifies the matrix family; marker shape identifies sizing; larger markers mean more positions. The red star and dotted lines mark original v11. Points above and to the right improve both plotted metrics. Black rings mark nondominated grid results; dashed lines join them when there is more than one. Connecting segments are not additional tested configurations. " + guide + " Repeated matrix aliases are removed; apparent clusters describe this in-sample search and do not establish robustness.", CHART_PERIOD)
    show_static(pareto_chart(chart_parameter_space, chart_original["metrics"], x_metric, y_metric), CHART_PERIOD)

### CAGR versus Sharpe — return and consistency — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one tested configuration. Colour identifies the matrix family; marker shape identifies sizing; larger markers mean more positions. The red star and dotted lines mark original v11. Points above and to the right improve both plotted metrics. Black rings mark nondominated grid results; dashed lines join them when there is more than one. Connecting segments are not additional tested configurations. The dashed frontier ignores drawdown; use the next two maps to inspect the depth of losses. Repeated matrix aliases are removed; apparent clusters describe this in-sample search and do not establish robustness.

### CAGR versus maximum drawdown — return and loss depth — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one tested configuration. Colour identifies the matrix family; marker shape identifies sizing; larger markers mean more positions. The red star and dotted lines mark original v11. Points above and to the right improve both plotted metrics. Black rings mark nondominated grid results; dashed lines join them when there is more than one. Connecting segments are not additional tested configurations. Drawdown is signed: −5% is better than −15%. Higher points have shallower losses. The dashed frontier ignores Sharpe. Repeated matrix aliases are removed; apparent clusters describe this in-sample search and do not establish robustness.

### Sharpe versus maximum drawdown — consistency and loss depth — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each point is one tested configuration. Colour identifies the matrix family; marker shape identifies sizing; larger markers mean more positions. The red star and dotted lines mark original v11. Points above and to the right improve both plotted metrics. Black rings mark nondominated grid results; dashed lines join them when there is more than one. Connecting segments are not additional tested configurations. Drawdown is signed: −5% is better than −15%. Higher points have shallower losses. The dashed frontier ignores CAGR. Repeated matrix aliases are removed; apparent clusters describe this in-sample search and do not establish robustness.

## Interactive parameter explorer

Filter combinations directly rather than comparing long result tables. This interactive chart requires Plotly JavaScript from its usual CDN; the preceding heatmaps and scatter maps remain static saved images.

In [21]:
chart_intro("Brush the axes to inspect a subset of configurations", "Each line is one of the distinct settings after recipe canonicalisation. Drag a range on an axis to filter, for example six vaults, inverse variance, missing=True and high Sharpe. Move axes to compare relationships; clear range selections to restore the full grid. CAGR, drawdown and cash are percentages; Sharpe is a ratio. Lines are coloured by Sharpe. Matrix labels identify the fixed clipping recipes listed above. Filtering does not run a new optimisation or create a validation sample.", CHART_PERIOD)
with_backtest_period(parallel_coordinates(chart_parameter_space), CHART_PERIOD).show(renderer="notebook_connected")

### Brush the axes to inspect a subset of configurations — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each line is one of the distinct settings after recipe canonicalisation. Drag a range on an axis to filter, for example six vaults, inverse variance, missing=True and high Sharpe. Move axes to compare relationships; clear range selections to restore the full grid. CAGR, drawdown and cash are percentages; Sharpe is a ratio. Lines are coloured by Sharpe. Matrix labels identify the fixed clipping recipes listed above. Filtering does not run a new optimisation or create a validation sample.

## Equity curves

Compare unique 2026 replay paths, including the original v11 and near-v6 controls. Equivalent matrix aliases appear once. The dollar-cap paths are in a separate panel. Both panels use the same USD axis and family colours.

In [22]:
equity_floor=min(float(run["equity"].min()) for run in chart_display_runs.values())*.95
equity_ceiling=max(float(run["equity"].max()) for run in chart_display_runs.values())*1.05
for allocator in ["parent","accepted_dollar_cap"]:
    frames=[]
    for label,run in chart_display_runs.items():
        if run["mode"]==allocator:
            frame=run["equity"].rename("Portfolio equity (USD)").rename_axis("Date").reset_index()
            frame["Strategy"]=run["family"]
            frame["Configuration"]=run_title(run)
            frames.append(frame)
    title="Strategy equity paths" if allocator=="parent" else "Accepted-dollar-cap equity paths"
    chart_intro(title,"Each replay starts from $150,000 on 1 January 2026 with no inherited positions. Curves are net of the source modelled redemption costs. Both panels share colours and the USD axis; configuration details are in the metrics table. Match each path's CAGR, Sharpe and drawdown to that table; a late rally can conceal earlier risk.", CHART_PERIOD)
    fig=px.line(pd.concat(frames),x="Date",y="Portfolio equity (USD)",color="Strategy",color_discrete_map=FAMILY_COLOURS,hover_data=["Configuration"],title=title)
    fig.update_layout(template="plotly_white",legend=dict(title="Strategy",font=dict(size=13)),margin=dict(r=270,t=100,b=70))
    fig.update_yaxes(range=[equity_floor,equity_ceiling],tickprefix="$",tickformat=",.0f")
    with_backtest_period(fig, CHART_PERIOD).show(width=1500,height=800)

### Strategy equity paths — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each replay starts from $150,000 on 1 January 2026 with no inherited positions. Curves are net of the source modelled redemption costs. Both panels share colours and the USD axis; configuration details are in the metrics table. Match each path's CAGR, Sharpe and drawdown to that table; a late rally can conceal earlier risk.

### Accepted-dollar-cap equity paths — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each replay starts from $150,000 on 1 January 2026 with no inherited positions. Curves are net of the source modelled redemption costs. Both panels share colours and the USD axis; configuration details are in the metrics table. Match each path's CAGR, Sharpe and drawdown to that table; a late rally can conceal earlier risk.

## Quarterly profit heatmaps

Use the existing trade-executor returns heatmap with calendar-quarter aggregation. Each cell shows compounded portfolio profit/loss in per cent, including changes in the value of open positions. The calculation uses the same daily returns as the metrics table.

All charts share a symmetric colour scale centred on zero, so a given colour represents the same return in every strategy. Q4 covers only 1–4 October and is labelled partial. Quarterly returns are not annualised.

In [23]:
from matplotlib import pyplot as plt
from tradeexecutor.visual.equity_curve import visualise_returns_over_time

quarter_limit=quarterly_colour_limit(chart_display_runs)
for label,run in chart_display_runs.items():
    title=f"Quarterly returns — {run['family']}"+ (" / dollar cap" if run["mode"]!="parent" else "")
    chart_intro(title,f"{run_title(run)}. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.", CHART_PERIOD)
    fig=visualise_returns_over_time(run["daily_returns"],period="quarter")
    style_quarterly_returns(fig,quarter_limit,title,CHART_PERIOD)
    display(fig)
    plt.close(fig)

### Quarterly returns — Original v11 — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Original v11 · 6 vaults · Inverse variance · missing=True. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — Near-v6 360/46 legacy — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Near-v6 360/46 legacy · 6 vaults · Inverse variance · missing=True. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — CAGR matrix — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

CAGR matrix · 6 vaults · Equal · missing=False. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — 50/50 blend — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

50/50 blend · 6 vaults · Equal · missing=False. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — 60/40 blend — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

60/40 blend · 6 vaults · Equal · missing=False. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — Dense equal matrix — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Dense equal matrix · 4 vaults · Score · missing=True. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — CAGR matrix / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

CAGR matrix / dollar cap · 6 vaults · Equal · missing=False. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — 60/40 blend / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

60/40 blend / dollar cap · 6 vaults · Equal · missing=False. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

### Quarterly returns — Sharpe matrix / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Sharpe matrix / dollar cap · 6 vaults · Inverse variance · missing=True. Each tile is compounded portfolio return for that calendar quarter, including cash. All strategies share the same colour scale, centred on zero. Only 2026 quarters are shown; Q4 is partial; quarterly returns are not annualised. Use the pattern to see whether performance is spread across time or concentrated in one rally.

## Absolute asset allocation

Each stacked chart shows the USD value of every vault holding and reserve cash. Values are absolute, rather than normalised to 100%.

All strategies share the same zero-based USD axis. The same vault keeps its colour across charts; reserve cash is grey. Colours identify holdings, not performance.

In [24]:
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset

allocation_figures=[]
for label, run in chart_display_runs.items():
    title=f"Absolute allocation — {run['family']}"+ (" / dollar cap" if run["mode"]!="parent" else "")
    result = run["result"]
    run_parameters = run["parameters"]
    charts = strategy.create_charts(
        timestamp=None,
        parameters=run_parameters,
        strategy_universe=result.strategy_universe,
        execution_context=notebook_execution_context,
    )
    if REUSE_SAVED_EXPERIMENT:
        from tradeexecutor.strategy.chart.definition import ChartInput
        asset_weights_fig=equity_curve_by_asset(ChartInput(
            execution_context=notebook_execution_context,
            state=result.state,pairs=[],
            backtest_start_at=run_parameters.backtest_start,
            backtest_end_at=run_parameters.backtest_end,
        ))
    else:
        chart_renderer = ChartBacktestRenderingSetup(
            registry=charts,
            pairs=[] if REUSE_SAVED_EXPERIMENT else None,
            strategy_input_indicators=result.indicators,
            state=result.state,
            backtest_start_at=run_parameters.backtest_start,
            backtest_end_at=run_parameters.backtest_end,
        )
        asset_weights_fig = chart_renderer.render(equity_curve_by_asset)
    asset_weights_fig.update_layout(title=title, width=1500, height=900)
    allocation_figures.append((title,run,asset_weights_fig))

colours=allocation_colours([fig for _,_,fig in allocation_figures])
for title,run,asset_weights_fig in allocation_figures:
    chart_intro(title,f"{run_title(run)}. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.", CHART_PERIOD)
    style_allocation(asset_weights_fig,colours,equity_ceiling)
    with_backtest_period(asset_weights_fig, CHART_PERIOD)
    # Keep every vault in the static legend visible in the 2026 replay.
    chart_height = max(900, 170 + 20 * len(asset_weights_fig.data))
    asset_weights_fig.show(width=1500, height=chart_height)

### Absolute allocation — Original v11 — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Original v11 · 6 vaults · Inverse variance · missing=True. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — Near-v6 360/46 legacy — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Near-v6 360/46 legacy · 6 vaults · Inverse variance · missing=True. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — CAGR matrix — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

CAGR matrix · 6 vaults · Equal · missing=False. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — 50/50 blend — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

50/50 blend · 6 vaults · Equal · missing=False. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — 60/40 blend — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

60/40 blend · 6 vaults · Equal · missing=False. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — Dense equal matrix — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Dense equal matrix · 4 vaults · Score · missing=True. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — CAGR matrix / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

CAGR matrix / dollar cap · 6 vaults · Equal · missing=False. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — 60/40 blend / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

60/40 blend / dollar cap · 6 vaults · Equal · missing=False. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

### Absolute allocation — Sharpe matrix / dollar cap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Sharpe matrix / dollar cap · 6 vaults · Inverse variance · missing=True. Stacked areas show marked USD amounts, including reserve cash; the top edge is portfolio equity. This is not a percentage allocation chart. Vault colours and USD axis bounds are shared across strategies; grey always identifies cash. The full vault legend is retained. Adjacent areas interpolate between valuation snapshots.

## Robustness analysis

Reconstruct each position's PnL from successful USDC trade flows and marked holdings, including final realised profit for closed positions. Reconcile the sum to equity minus the initial deposit, then group by vault address. Show the top vaults and positions and attribute the three strongest portfolio days.

Use the repository's `curator.py` classifications and quarantine windows to annotate influential vaults. Binance BTC and ETH opening prices provide broad-market context over the matching prior-snapshot-to-current-snapshot UTC interval. With two-day decisions, a return booked on one date can cover more than one calendar day; compare that interval rather than a single market day. The best-day-zero and leading-vault-PnL removal figures are fixed-path sensitivities, not executable alternative strategies; they do not establish out-of-sample performance. Cash exposure alone cannot separate reduced market exposure from better sizing.

All tables in this section use fresh 2026 states: **2026-01-01 → 2026-10-05 (end exclusive, UTC)**.

In [25]:
chart_view_labels={label:run_title(run) for label,run in chart_display_runs.items()}
from tradingstrategy.binance.price import fetch_binance_price
from tradeexecutor.curator.curator import EXCLUDED_VAULTS, get_hyperliquid_concentration_tweak, is_quarantined

# Incrementally cached Binance reference series, used only for best-day context.
market_open_prices = {}
for symbol in ("BTCUSDT", "ETHUSDT"):
    prices = fetch_binance_price(symbol=symbol, start=chart_start - datetime.timedelta(days=7))
    market_open_prices[symbol] = prices["open"]

curated_quality = get_hyperliquid_concentration_tweak()
robustness_rows = []
contributor_tables = []
jump_tables = []
for label, run in chart_runs.items():
    state = run["result"].state
    daily = run["daily_returns"]
    daily_equity = run["equity"].resample("1D").last().reindex(daily.index).ffill()
    curves, position_metadata = calculate_position_profit_curves(state, daily.index)
    portfolio_pnl = daily_equity - float(run["parameters"].initial_cash)
    residual = curves.sum(axis=1) - portfolio_pnl
    assert residual.abs().max() < 0.01, f"Position attribution does not reconcile: {label}, {residual.abs().max()}"
    vault_curves = {}
    for address, group in position_metadata.groupby("Address"):
        vault_curves[address] = curves[group["Position id"].tolist()].sum(axis=1)
    vault_curves = pd.DataFrame(vault_curves)
    final_profits = vault_curves.iloc[-1].sort_values(ascending=False)
    leader = final_profits.index[0]
    leader_profit = float(final_profits.iloc[0])
    positive_profit_total = final_profits.clip(lower=0.0).sum()
    best_day = daily.idxmax()
    without_best_day = daily.copy()
    without_best_day.loc[best_day] = 0.0
    # Fixed-ledger attribution sensitivity: retain the original trades and sizing.
    without_leader_equity = daily_equity - vault_curves[leader]
    without_leader_returns = calculate_returns(without_leader_equity)
    robustness_rows.append({
        "Run": label, "Best day": best_day, "Best day return": float(daily.loc[best_day]),
        "Return kurtosis (excess)": float(daily.kurt()),
        "CAGR with best day set to zero": float(calculate_cagr(without_best_day)),
        "Sharpe with best day set to zero": float(calculate_sharpe(without_best_day, periods=365)),
        "Leading vault": position_metadata.loc[position_metadata.Address.eq(leader), "Vault"].iloc[0],
        "Leading vault profit (USD)": leader_profit,
        "Leading vault share of positive vault profits": float(leader_profit / positive_profit_total),
        "CAGR without leader PnL (fixed ledger)": float(calculate_cagr(without_leader_returns)),
        "Sharpe without leader PnL (fixed ledger)": float(calculate_sharpe(without_leader_returns, periods=365)),
        "Max attribution residual (USD)": float(residual.abs().max()),
    })
    names = position_metadata.groupby("Address").Vault.first()
    contribution = pd.DataFrame({"Address": final_profits.index, "Profit (USD)": final_profits.values})
    contribution["Vault"] = contribution.Address.map(names)
    contribution["Run"] = label
    contribution["Curated quality"] = contribution.Address.map(lambda address: getattr(curated_quality.get(address), "value", "unrated"))
    contribution["Curator excluded"] = contribution.Address.isin(EXCLUDED_VAULTS)
    contributor_tables.append(contribution.head(5))
    position_metadata["Run"] = label
    run["position_metadata"] = position_metadata
    run["vault_profit_curves"] = vault_curves
    daily_pnl = vault_curves.diff().fillna(vault_curves.iloc[0])
    top_days = daily.nlargest(3).index
    for day in top_days:
        gains = daily_pnl.loc[day].sort_values(ascending=False)
        positive = gains.clip(lower=0.0).sum()
        top_address = gains.index[0]
        snapshot_dates = pd.DatetimeIndex(run["equity"].index).normalize().unique()
        previous_snapshot = snapshot_dates[snapshot_dates < day][-1]
        previous_equity = float(daily_equity.loc[previous_snapshot])
        market_interval_returns = {
            symbol: float(series.loc[day] / series.loc[previous_snapshot] - 1.0)
            for symbol, series in market_open_prices.items()
        }
        jump_tables.append({
            "Run": label, "Date": day, "Portfolio return": float(daily.loc[day]),
            "Largest contributing vault": names[top_address],
            "Vault PnL that day (USD)": float(gains.iloc[0]),
            "Vault contribution / prior equity": float(gains.iloc[0] / previous_equity),
            "Share of positive vault PnL": float(gains.iloc[0] / positive) if positive else 0.0,
            "Previous snapshot": previous_snapshot,
            "BTC over snapshot interval": market_interval_returns["BTCUSDT"],
            "ETH over snapshot interval": market_interval_returns["ETHUSDT"],
            "Vault quarantined that day": is_quarantined(top_address, day.to_pydatetime()),
        })
robustness_df = pd.DataFrame(robustness_rows)
contribution_df = pd.concat(contributor_tables, ignore_index=True)
jumps_df = pd.DataFrame(jump_tables)
robust_view=robustness_df[robustness_df.Run.isin(chart_display_runs)].copy()
robust_view["Run"]=robust_view.Run.map(chart_view_labels)
percent_columns=[c for c in robust_view if "return" in c.lower() or "CAGR" in c or "share" in c]
robust_style=robust_view.style.format({**{c:"{:.2%}" for c in percent_columns},**{c:"{:.2f}" for c in robust_view if "Sharpe" in c or "kurtosis" in c}})
present_table(robust_style.background_gradient(subset=["Sharpe with best day set to zero","Sharpe without leader PnL (fixed ledger)"],cmap="RdYlGn").set_caption(f"Attribution sensitivities — {CHART_PERIOD}"), "Dependence on the best day and leading vault", "Remove the best daily return or the leading vault PnL while retaining the original trading path. These sensitivities show profit concentration, not executable exclusion strategies. Position PnL must reconcile to portfolio equity.", CHART_PERIOD)
present_table(contribution_df[contribution_df.Run.isin(chart_display_runs)].assign(Run=lambda d:d.Run.map(chart_view_labels)), "Top five vault contributors", "Rank vaults by total realised and marked PnL in each distinct 2026 strategy path. Curator labels provide context; they do not change these replays.", CHART_PERIOD, collapsed=True)
present_table(jumps_df[jumps_df.Run.isin(chart_display_runs)].assign(Run=lambda d:d.Run.map(chart_view_labels)), "Largest profit days and market context", "Attribute the three strongest booked returns in each strategy to vault PnL. BTC and ETH returns use the same previous-to-current valuation interval, which can span two days.", CHART_PERIOD, collapsed=True)

position_tables = []
for label, run in chart_runs.items():
    table = run["position_metadata"].sort_values("Final profit (USD)", ascending=False).head(3)
    position_tables.append(table)
present_table(pd.concat(position_tables, ignore_index=True).loc[lambda d:d.Run.isin(chart_display_runs)].assign(Run=lambda d:d.Run.map(chart_view_labels)), "Top three individual positions", "Rank individual positions by final realised and marked profit. Multiple positions can belong to the same vault, unlike the vault-level contributor table.", CHART_PERIOD, collapsed=True)
robustness_df.to_csv(ARTIFACT_DIR / "chart-2026-robustness.csv", index=False)
contribution_df.to_csv(ARTIFACT_DIR / "chart-2026-contributors.csv", index=False)
jumps_df.to_csv(ARTIFACT_DIR / "chart-2026-jumps.csv", index=False)
pd.concat(position_tables, ignore_index=True).to_csv(ARTIFACT_DIR / "chart-2026-top-positions.csv", index=False)


### Dependence on the best day and leading vault — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Remove the best daily return or the leading vault PnL while retaining the original trading path. These sensitivities show profit concentration, not executable exclusion strategies. Position PnL must reconcile to portfolio equity.

,Run,Best day,Best day return,Return kurtosis (excess),CAGR with best day set to zero,Sharpe with best day set to zero,Leading vault,Leading vault profit (USD),Leading vault share of positive vault profits,CAGR without leader PnL (fixed ledger),Sharpe without leader PnL (fixed ledger),Max attribution residual (USD)
1,50/50 blend · 6 vaults · Equal · missing=False,2026-06-26 00:00:00,4.37%,6.65,23.55%,1.24,Realist Capital,13979.886454,26.49%,17.75%,0.98,0.000000
2,60/40 blend · 6 vaults · Equal · missing=False,2026-06-26 00:00:00,4.58%,7.03,16.76%,0.94,Realist Capital,12423.561573,27.43%,12.48%,0.74,0.000000
3,CAGR matrix · 6 vaults · Equal · missing=False,2026-06-26 00:00:00,6.40%,16.86,-3.86%,-0.15,Realist Capital,8668.927050,29.75%,-3.30%,-0.12,0.000000
5,Dense equal matrix · 4 vaults · Score · missing=True,2026-01-07 00:00:00,3.06%,40.71,-1.99%,-0.21,Hyperliquidity Provider (HLP),2233.837633,39.90%,0.02%,0.05,0.000000
7,Original v11 · 6 vaults · Inverse variance · missing=True,2026-06-26 00:00:00,6.89%,13.67,29.30%,1.62,Realist Capital,24196.229340,44.77%,18.45%,1.10,0.000000
9,Near-v6 360/46 legacy · 6 vaults · Inverse variance · missing=True,2026-06-26 00:00:00,6.87%,13.50,28.76%,1.60,Realist Capital,25883.020549,48.48%,16.31%,0.99,0.000000
10,CAGR matrix / dollar cap · 6 vaults · Equal · missing=False,2026-06-26 00:00:00,4.16%,15.47,-5.52%,-0.28,Goon Edging,5186.743783,22.78%,-4.84%,-0.25,0.000000
11,Sharpe matrix / dollar cap · 6 vaults · Inverse variance · missing=True,2026-02-28 00:00:00,1.87%,41.21,5.58%,0.90,pmalt,3823.760114,28.42%,4.78%,0.75,0.000000
12,60/40 blend / dollar cap · 6 vaults · Equal · missing=False,2026-06-26 00:00:00,4.16%,7.38,12.10%,0.78,Realist Capital,8788.755329,23.10%,10.29%,0.68,0.000000


### Top five vault contributors — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Rank vaults by total realised and marked PnL in each distinct 2026 strategy path. Curator labels provide context; they do not change these replays.

Address,Profit (USD),Vault,Run,Curated quality,Curator excluded
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,13979.886454,Realist Capital,50/50 blend · 6 vaults · Equal · missing=False,unrated,False
0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,9900.708669,Goon Edging,50/50 blend · 6 vaults · Equal · missing=False,unrated,False
0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,9339.270096,DOEZOE,50/50 blend · 6 vaults · Equal · missing=False,unrated,False
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,6754.905876,Hyperrr,50/50 blend · 6 vaults · Equal · missing=False,unrated,False
0x07fd993f0fa3a185f7207adccd29f7a87404689d,3180.338440,[ Systemic Strategies ] L/S Grids,50/50 blend · 6 vaults · Equal · missing=False,high,False
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,12423.561573,Realist Capital,60/40 blend · 6 vaults · Equal · missing=False,unrated,False
0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,10904.013925,Goon Edging,60/40 blend · 6 vaults · Equal · missing=False,unrated,False
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,6744.711035,Hyperrr,60/40 blend · 6 vaults · Equal · missing=False,unrated,False
0x07fd993f0fa3a185f7207adccd29f7a87404689d,3322.048812,[ Systemic Strategies ] L/S Grids,60/40 blend · 6 vaults · Equal · missing=False,high,False
0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2978.394458,Citadel,60/40 blend · 6 vaults · Equal · missing=False,unrated,False


### Largest profit days and market context — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Attribute the three strongest booked returns in each strategy to vault PnL. BTC and ETH returns use the same previous-to-current valuation interval, which can span two days.

Run,Date,Portfolio return,Largest contributing vault,Vault PnL that day (USD),Vault contribution / prior equity,Share of positive vault PnL,Previous snapshot,BTC over snapshot interval,ETH over snapshot interval,Vault quarantined that day
50/50 blend · 6 vaults · Equal · missing=False,2026-06-26,0.043744,Realist Capital,4133.359673,0.024998,0.529626,2026-06-24,-0.046863,-0.059545,False
50/50 blend · 6 vaults · Equal · missing=False,2026-04-09,0.036903,Hyperrr,5350.210255,0.031707,0.856826,2026-04-07,0.032188,0.039314,False
50/50 blend · 6 vaults · Equal · missing=False,2026-09-22,0.032303,DOEZOE,4399.259410,0.024940,0.771673,2026-09-20,0.066093,0.054507,False
60/40 blend · 6 vaults · Equal · missing=False,2026-06-26,0.045848,Realist Capital,3970.478402,0.024252,0.510454,2026-06-24,-0.046863,-0.059545,False
60/40 blend · 6 vaults · Equal · missing=False,2026-04-09,0.036901,Hyperrr,5343.884995,0.031700,0.856681,2026-04-07,0.032188,0.039314,False
60/40 blend · 6 vaults · Equal · missing=False,2026-09-26,0.032765,DOEZOE,5224.424592,0.030137,0.895748,2026-09-24,-0.003526,0.002578,False
CAGR matrix · 6 vaults · Equal · missing=False,2026-06-26,0.063999,Realist Capital,5661.696958,0.039631,0.601351,2026-06-24,-0.046863,-0.059545,False
CAGR matrix · 6 vaults · Equal · missing=False,2026-05-17,0.031271,Realist Capital,2351.727841,0.015811,0.476966,2026-05-15,-0.036280,-0.044888,False
CAGR matrix · 6 vaults · Equal · missing=False,2026-02-28,0.030214,pmalt,2957.492987,0.019097,0.631960,2026-02-26,-0.031122,-0.062037,False
Dense equal matrix · 4 vaults · Score · missing=True,2026-01-07,0.030640,Citadel,3488.162034,0.023271,0.759484,2026-01-05,0.024235,0.048373,False


### Top three individual positions — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Rank individual positions by final realised and marked profit. Multiple positions can belong to the same vault, unlike the vault-level contributor table.

Position id,Vault,Address,Opened,Closed,Final profit (USD),Run
96,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,2026-09-18,NaT,12243.582552,50/50 blend · 6 vaults · Equal · missing=False
19,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-31,2026-04-21,11965.140611,50/50 blend · 6 vaults · Equal · missing=False
55,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-24,2026-08-21,8127.265694,50/50 blend · 6 vaults · Equal · missing=False
21,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-31,2026-04-21,11953.455976,60/40 blend · 6 vaults · Equal · missing=False
60,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-24,2026-08-21,8041.718485,60/40 blend · 6 vaults · Equal · missing=False
43,Goon Edging,0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,2026-05-25,2026-07-22,6972.339005,60/40 blend · 6 vaults · Equal · missing=False
22,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-31,2026-04-03,12576.617737,CAGR matrix · 6 vaults · Equal · missing=False
80,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-24,2026-08-21,10728.690742,CAGR matrix · 6 vaults · Equal · missing=False
59,Goon Edging,0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,2026-05-25,2026-07-20,6557.532934,CAGR matrix · 6 vaults · Equal · missing=False
6,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2026-01-27,NaT,8104.750495,Dense equal matrix · 4 vaults · Score · missing=True


## Missing-score, information and allocation audits

An unscored vault can be funded with missing=True under the unchanged source sizing semantics. The missing=False policy retains the previous zero-total-score restriction.

All tables in this section use fresh 2026 states: **2026-01-01 → 2026-10-05 (end exclusive, UTC)**.

In [26]:
audit=[]
for label,run in chart_runs.items():
    ledger=run["allocation_df"]
    if not len(ledger):continue
    funded=ledger[ledger["Final target (USD)"].gt(0)]
    if not run["missing"]:
        assert not (funded["Ranking score"].fillna(0)<=0).any()
    observations=[]
    for row in funded.to_dict("records"):
        i=panel.date_index.get(pd.Timestamp(row["Date"]))
        j=panel.pair_index.get(int(row["Pair id"]))
        if i is None or j is None:continue
        observations.append({"Target":row["Final target (USD)"],"1y":panel.valid[i,j,list(PERIODS).index("1y")],
                             "2y":panel.valid[i,j,list(PERIODS).index("2y")],"Flat":panel.valid[i,j,0] and panel.nonflat[i,j,0]==0,
                             "Unscored":bool(row["Unscored"])})
    t=pd.DataFrame(observations)
    if len(t):
        total=t.Target.sum()
        audit.append({"Run":label,"Funded target rows":len(t),"Unscored target dollars / target dollars":t.loc[t.Unscored,"Target"].sum()/total,
                      "Missing 1y target dollars / target dollars":t.loc[~t["1y"],"Target"].sum()/total,
                      "Missing 2y target dollars / target dollars":t.loc[~t["2y"],"Target"].sum()/total,
                      "Flat 1m target dollars / target dollars":t.loc[t.Flat,"Target"].sum()/total})
audit_df=pd.DataFrame(audit)
audit_view=audit_df[audit_df.Run.isin(chart_display_runs)].copy()
audit_view["Run"]=audit_view.Run.map(chart_view_labels)
fraction_columns=[c for c in audit_view if "/ target dollars" in c]
present_table(audit_view.style.format({c:"{:.1%}" for c in fraction_columns}).background_gradient(subset=fraction_columns,cmap="Blues").set_caption(f"Funded information gaps — {CHART_PERIOD}"), "Information available when allocating capital", "Fractions weight final target dollars across 2026 decisions. Darker blue means more target dollars with missing scores, incomplete lookback history or flat 1-month marks. These are not realised exposure or return metrics; original v11 has no matrix allocation ledger.", CHART_PERIOD)
audit_df.to_csv(ARTIFACT_DIR / "chart-2026-funded-information.csv",index=False)
losers=pd.concat([r["position_metadata"].sort_values("Final profit (USD)").head(3) for r in chart_runs.values()],ignore_index=True)
present_table(losers[losers.Run.isin(chart_display_runs)].assign(Run=lambda d:d.Run.map(chart_view_labels)), "Three worst individual positions", "Lowest final realised and marked position PnL in each distinct 2026 path. Positive values can occur if a path has fewer than three losing positions.", CHART_PERIOD, collapsed=True)
losers.to_csv(ARTIFACT_DIR / "chart-2026-worst-positions.csv",index=False)


### Information available when allocating capital — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fractions weight final target dollars across 2026 decisions. Darker blue means more target dollars with missing scores, incomplete lookback history or flat 1-month marks. These are not realised exposure or return metrics; original v11 has no matrix allocation ledger.

,Run,Funded target rows,Unscored target dollars / target dollars,Missing 1y target dollars / target dollars,Missing 2y target dollars / target dollars,Flat 1m target dollars / target dollars
1,50/50 blend · 6 vaults · Equal · missing=False,834,0.0%,0.0%,74.7%,0.0%
2,60/40 blend · 6 vaults · Equal · missing=False,834,0.0%,0.0%,74.3%,0.0%
3,CAGR matrix · 6 vaults · Equal · missing=False,834,0.0%,0.0%,74.9%,0.0%
5,Dense equal matrix · 4 vaults · Score · missing=True,555,0.0%,0.0%,0.0%,0.0%
8,Near-v6 360/46 legacy · 6 vaults · Inverse variance · missing=True,834,0.0%,0.0%,73.8%,0.0%
9,CAGR matrix / dollar cap · 6 vaults · Equal · missing=False,834,0.0%,0.0%,76.6%,0.0%
10,Sharpe matrix / dollar cap · 6 vaults · Inverse variance · missing=True,834,0.0%,0.0%,64.0%,0.0%
11,60/40 blend / dollar cap · 6 vaults · Equal · missing=False,834,0.0%,0.0%,76.7%,0.0%


### Three worst individual positions — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Lowest final realised and marked position PnL in each distinct 2026 path. Positive values can occur if a path has fewer than three losing positions.

Position id,Vault,Address,Opened,Closed,Final profit (USD),Run
16,gaspedal,0xff9152cce6fbd30988b1aef70df6f086c99e2c55,2026-01-29,2026-01-31,-5130.642023,50/50 blend · 6 vaults · Equal · missing=False
59,Pepe vs,0x2ee1f7d5650bb9bc08e101dc5ab2300b4f5c1be9,2026-07-06,2026-07-08,-4711.512785,50/50 blend · 6 vaults · Equal · missing=False
1,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-01,2026-01-21,-4166.646194,50/50 blend · 6 vaults · Equal · missing=False
18,gaspedal,0xff9152cce6fbd30988b1aef70df6f086c99e2c55,2026-01-29,2026-01-31,-5126.266328,60/40 blend · 6 vaults · Equal · missing=False
63,Pepe vs,0x2ee1f7d5650bb9bc08e101dc5ab2300b4f5c1be9,2026-07-06,2026-07-08,-4711.512785,60/40 blend · 6 vaults · Equal · missing=False
1,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-01,2026-01-21,-4166.242602,60/40 blend · 6 vaults · Equal · missing=False
34,500 to 1M,0x3bd6460a3782a80fa9331e65e4c2952795a86228,2026-03-08,2026-03-14,-8721.041025,CAGR matrix · 6 vaults · Equal · missing=False
42,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-04-09,2026-04-21,-6204.465211,CAGR matrix · 6 vaults · Equal · missing=False
81,Pepe vs,0x2ee1f7d5650bb9bc08e101dc5ab2300b4f5c1be9,2026-07-06,2026-07-08,-4711.512785,CAGR matrix · 6 vaults · Equal · missing=False
2,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2026-01-01,2026-01-21,-7795.351346,Dense equal matrix · 4 vaults · Score · missing=True
